# 1. CrowdHumanを用いたYOLO追加学習
vbox=可視人物、hbox=頭部、both=person/headの2クラスです。モデルは参考実装のYOLO26nを使用します。
[公式配布・条件](https://www.crowdhuman.org/download.html) は非商用の研究・教育用途に限定し、画像再配布を禁止しています。
[Ultralyticsライセンス](https://www.ultralytics.com/license) も確認してください。

CrowdHumanの元ZIP・ODGTは非公開のGoogle Driveフォルダへ保存し、実行時に `/content/crowdhuman/` へコピーします。
展開・変換・学習・評価はColabの一時ストレージで行います。Driveフォルダの共有リンクを公開しないでください。
ColabはGoogleのクラウドです。利用条件でクラウド処理を認められない場合はこのノートブックを実行せず、ローカル用に別途移植してください。
ランタイム終了・リセットでColab上のコピーと重みが失われますが、Driveに保管した元ファイルは残ります。
画像・アノテーションの表示・公開・再配布は行いません。
Colabの「編集 → ノートブックの設定 → 保存時にコードセルの出力を除外する」を有効にしてください。
このノートブック自身にも出力除外メタデータを設定しています。共有前に全出力を消してください。

GPU学習・実データ・ブラウザアプリとの接続は開発環境では未検証です。

## 2. Colab実行環境とGPUの確認
Colabの「ランタイム → ランタイムのタイプを変更」でGPUを選択します。GPUがなければ警告し、学習時に停止します。

In [ ]:
import os, sys, subprocess, warnings
from pathlib import Path
if 'google.colab' not in sys.modules:
    try:
        import google.colab
    except ImportError:
        raise RuntimeError('Google Colabで実行してください') from None
BOOTSTRAP = Path('/content/crowdhuman')
if BOOTSTRAP.resolve() != BOOTSTRAP:
    raise RuntimeError('保存先が別の場所へリンクされています')
BOOTSTRAP.mkdir(parents=True, exist_ok=True)
os.environ.update(YOLO_CONFIG_DIR=str(BOOTSTRAP / 'config'), YOLO_OFFLINE='true',
                  YOLO_AUTOINSTALL='false', YOLO_VERBOSE='false', WANDB_MODE='disabled',
                  WANDB_DISABLED='true', COMET_MODE='DISABLED', HF_HUB_OFFLINE='1')
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True) if __import__('shutil').which('nvidia-smi') else None
if gpu is None or gpu.returncode:
    warnings.warn('GPUが見つかりません。GPUランタイムを選択してください。')
else:
    print(gpu.stdout.strip())

## 3. ライブラリのインストール
Ultralyticsを8.4.52に固定します。ColabのCUDA対応PyTorchを利用し、実際の依存バージョンは実験に記録します。
インストールログはランタイム内に保存します。依存を変更した場合はランタイムを再起動して最初から実行してください。

In [ ]:
REQUIREMENTS = '# ColabのCUDA対応torch/torchvisionを維持。実際の全バージョンは実行時に記録。\nultralytics==8.4.52\nPillow>=10,<13\nPyYAML>=6,<7\nonnx>=1.17,<2\nonnxruntime>=1.20,<2\n'
requirements_path = BOOTSTRAP / 'requirements.txt'
requirements_path.write_text(REQUIREMENTS)
with open(BOOTSTRAP / 'install.log', 'w') as log:
    installed = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements_path)], stdout=log, stderr=log)
if installed.returncode:
    raise RuntimeError('インストール失敗。ランタイム内のinstall.logを確認してください。')
print('インストール完了')

## 4. 学習設定
初回は `DEBUG_MODE=True`、少ないepochで動作確認してください。正式比較ではFalseに戻し、別のRUN_NAMEを使います。
`MODES_TO_RUN=["vbox", "hbox"]` で同じ設定の比較を一括実行できます。bothは2クラスです。
利用条件とクラウド処理の可否を確認後に `TERMS_REVIEWED=True` にします。認証情報は入力しません。
Driveの保管先をDRIVE_DATASET_DIRに指定します。初回はHugging Faceから不足ファイルだけをDriveへ保存し、次回から再利用します。
既に元ファイルをDriveへ配置した場合はDOWNLOAD_MISSING_TO_DRIVE=Falseにできます。HF_DATASET_REVISIONでデータの版を固定します。
手順12の保存から再開する場合は `RESTORE_PREPARED_FROM_DRIVE=True` にし、手順2〜5の後に13へ進みます。
保存時と同じMODES_TO_RUN・DEBUG_MODE・DEBUG_LIMITを指定してください。学習前ならRESUME=Falseのままです。
中断した同一実験だけ `RESUME=True` にします。設定・データ・依存バージョンが変わる再開は拒否します。

In [ ]:
TRAIN_MODE = 'vbox'
MODES_TO_RUN = [TRAIN_MODE]  # 比較例: ['vbox', 'hbox']; 3モード: ['vbox', 'hbox', 'both']
BASE_MODEL = 'yolo26n.pt'
EPOCHS = 50
BATCH_SIZE = 16
IMAGE_SIZE = 640
SEED = 42
DEBUG_MODE = False
DEBUG_LIMIT = 32  # 各splitの先頭N画像。すべてのモードで同じ画像。
RUN_NAME = 'experiment'
RESUME = False  # 中断から再開する場合はTrue。Driveの最新保存を自動復元。
SAVE_EVERY_EPOCH_TO_DRIVE = True
DRIVE_CHECKPOINT_DIR = '/content/drive/MyDrive/crowdhuman/checkpoints'
DRIVE_PREPARED_DIR = '/content/drive/MyDrive/crowdhuman/prepared'
RESTORE_PREPARED_FROM_DRIVE = False  # 手順2〜5の後、13から復元・再開する場合True
ALLOW_CPU = False
TERMS_REVIEWED = False
DRIVE_MOUNT_POINT = '/content/drive'
DRIVE_DATASET_DIR = '/content/drive/MyDrive/crowdhuman'
DOWNLOAD_MISSING_TO_DRIVE = True
HF_DATASET_REPO = 'sshao0516/CrowdHuman'
HF_DATASET_REVISION = 'd97203da87e348ea69f7a7633a57c21a956120a6'
EXPORT_ONNX = True
WEBAPP_CONTRACT = None  # 既存アプリから採取したinput/outputs辞書。未指定では互換性を確定しません。
BASE_DIR = '/content/crowdhuman'
RAW_DIR = f'{BASE_DIR}/raw'
PROCESSED_DIR = f'{BASE_DIR}/processed'
RUNS_DIR = f'{BASE_DIR}/runs'
EXPORT_DIR = f'{BASE_DIR}/exports'
assert TRAIN_MODE in ('vbox', 'hbox', 'both')
assert MODES_TO_RUN and len(set(MODES_TO_RUN)) == len(MODES_TO_RUN)
assert all(mode in ('vbox', 'hbox', 'both') for mode in MODES_TO_RUN)

## 5. ディレクトリと非公開設定
以下は通常のPython関数です。ノートブック内で直接確認・編集でき、srcファイルの取得・生成・importは不要です。
raw/archives・raw/images・raw/annotations・processed・runs・exportsを作成します。
外部追跡サービス・同期・analyticsのコールバックを無効にし、学習ログもこの配下だけに保存します。
GitHubで管理する開発用srcとこのノートブックは、tools/build_notebook.pyで同期できます。Colabでの実行にはこの.ipynbだけを使います。

In [ ]:
"""Enforce runtime storage and disable integrations before importing YOLO."""
import os
import contextlib
import logging
from contextvars import ContextVar
from pathlib import Path

BASE = Path('/content/crowdhuman')
_DOWNLOAD_STORAGE = ContextVar('crowdhuman_local_download_storage', default=None)


def storage_root():
    return _DOWNLOAD_STORAGE.get() or BASE


@contextlib.contextmanager
def local_download_storage(directory):
    """Explicit private cache scope for local/Drive storage; preserve the Colab default."""
    root = Path(directory).expanduser().absolute()
    if root.resolve() != root:
        raise ValueError('Local download directory must not traverse symlinks')
    token = _DOWNLOAD_STORAGE.set(root)
    try:
        yield root
    finally:
        _DOWNLOAD_STORAGE.reset(token)


def runtime_path(path):
    path = Path(path).absolute()
    resolved = path.resolve()
    root = storage_root()
    if resolved != root and root not in resolved.parents:
        raise ValueError(f'CrowdHuman paths must stay under {root}')
    return path


def initialize(base=None):
    base = runtime_path(BASE if base is None else base)
    if base != BASE:
        raise ValueError('BASE_DIR must be /content/crowdhuman')
    for part in ('raw/archives', 'raw/images', 'raw/annotations', 'processed', 'runs', 'exports', 'weights', 'config'):
        runtime_path(base / part).mkdir(parents=True, exist_ok=True)
    os.environ.update(YOLO_CONFIG_DIR=str(base / 'config'), YOLO_OFFLINE='true',
                      YOLO_AUTOINSTALL='false', YOLO_VERBOSE='false',
                      WANDB_MODE='disabled', COMET_MODE='DISABLED',
                      WANDB_DISABLED='true', HF_HUB_OFFLINE='1')
    return base


def private_yolo():
    initialize()
    from ultralytics import YOLO, settings
    keys = ('sync', 'hub', 'wandb', 'mlflow', 'clearml', 'comet', 'dvc', 'neptune', 'raytune', 'tensorboard')
    settings.update({key: False for key in keys if key in settings})
    settings.update({'datasets_dir': str(BASE / 'processed'),
                     'weights_dir': str(BASE / 'weights'), 'runs_dir': str(BASE / 'runs')})
    # Disable integration registration, including Platform and analytics callbacks.
    from ultralytics.utils import callbacks
    callbacks.add_integration_callbacks = lambda instance: None
    from ultralytics.utils.callbacks import base
    base.add_integration_callbacks = lambda instance: None
    return YOLO


@contextlib.contextmanager
def local_console(path):
    """Capture pre-existing YOLO log handlers as well as stdout/progress bars."""
    path = runtime_path(path)
    logger = logging.getLogger('ultralytics')
    old_handlers, old_propagate = logger.handlers[:], logger.propagate
    with open(path, 'a') as log:
        logger.handlers = [logging.StreamHandler(log)]
        logger.propagate = False
        try:
            with contextlib.redirect_stdout(log), contextlib.redirect_stderr(log):
                yield
        finally:
            logger.handlers, logger.propagate = old_handlers, old_propagate

GITIGNORE_CONTENT = 'CrowdHuman/\ncrowdhuman/\ndatasets/\ndata/\n*.odgt\n*.zip\n*.jpg\n*.jpeg\n*.png\n*.webp\nruns/\nweights/\ncheckpoints/\nexports/\n*.pt\n*.pth\n*.onnx\n.env\n.env.*\n*.key\n__pycache__/\n.ipynb_checkpoints/\n.venv/\n.pytest_cache/\n*.cache\n\n# Runtime diagnostic logs\nlogs/\n*.log\n'
BASE = initialize(Path(BASE_DIR))
with local_console(BASE / 'setup.log'):
    YOLO = private_yolo()
print('ランタイム内の保存先とプライバシー設定を準備しました')

## 6. Google Driveへ保管し、/contentへコピー
Google Driveをマウントし、非公開の `MyDrive/crowdhuman/` に元の画像ZIP4個とODGT2個を保管します。
初回は[sshao0516/CrowdHuman](https://huggingface.co/datasets/sshao0516/CrowdHuman)から不足分だけをDriveへ取得します。
既にファイルがある場合は `raw/archives/` にZIP、`raw/annotations/` にODGTを配置してください。
次に6ファイルを `/content/crowdhuman/raw/` へコピーし、SHA-256・ZIP CRC・ODGT形式を確認します。
同じセッション内の正常なローカルコピーはDriveと一致すれば再利用します。内容の異なるローカルファイルは無言で上書きしません。
コピー・照合・検証の進捗は1つの表示欄に出力します。取得ログはDrive側の `logs/download.jsonl`、コピーログはColab側の `logs/drive_copy.jsonl` に保存します。
画像展開・ラベル変換・学習はこの後のセルでローカルコピーに対して実行します。共有・公開や外部追跡への送信は行いません。

In [ ]:
"""Public Hugging Face downloads, local validation and safe extraction."""
import json
import sys
import time
import traceback
import uuid
from datetime import datetime, timezone
import re
import shutil
import stat
import urllib.request
import urllib.error
from urllib.parse import urlsplit
import zipfile
import zlib
from pathlib import Path

HF_REPO_ID = 'sshao0516/CrowdHuman'
HF_REVISION = 'd97203da87e348ea69f7a7633a57c21a956120a6'
ARCHIVES = ('CrowdHuman_train01.zip', 'CrowdHuman_train02.zip', 'CrowdHuman_train03.zip', 'CrowdHuman_val.zip')
ANNOTATIONS = ('annotation_train.odgt', 'annotation_val.odgt')
EXPECTED = {'train': 15000, 'val': 4370}


class DownloadProgress:
    """Throttled text progress for a terminal or one updating notebook display."""
    def __init__(self, stream=None, interval=1.0, notebook=False):
        self.stream = sys.stderr if stream is None else stream
        self.interval = interval
        self.notebook = notebook
        self.last_update = None
        self.task = None
        self.started = None
        self.display = None

    def __call__(self, event):
        now = time.monotonic()
        stage = event['stage']
        task = (stage, event.get('filename'))
        changed = task != self.task or event.get('processed') == 0
        if changed:
            self.task, self.started = task, now
        finished = event.get('finished', False) or stage in ('reused', 'file_complete', 'complete', 'error', 'interrupted')
        if not changed and not finished and self.last_update is not None and now - self.last_update < self.interval:
            return
        self.last_update = now
        done = event.get('processed', 0)
        total = event.get('total', 0)
        prefix = f"[{event.get('completed_files', 0)}/{event.get('file_total', 6)} 完了]"
        filename = event.get('filename', '')
        labels = {'request': '接続中', 'validation': '検証中', 'transfer': 'ダウンロード中',
                  'copy': 'Driveからコピー中', 'checksum': 'チェックサム照合中',
                  'reused': '検証OK・再利用', 'file_complete': '取得・検証完了',
                  'complete': '全ファイル準備完了', 'error': '失敗', 'interrupted': '中断'}
        text = f"{prefix} {labels.get(stage, stage)} {filename}"
        if stage in ('transfer', 'validation', 'copy', 'checksum'):
            percent = f'{100 * done / total:5.1f}%' if total else '  -- %'
            elapsed = max(0.0, now - self.started)
            speed = done / elapsed if elapsed else 0
            remaining = max(0, total - done) / speed if total and speed else None
            eta = f'{int(remaining)//60:02d}:{int(remaining)%60:02d}' if remaining is not None else '--:--'
            if event.get('unit') == 'records':
                amount = f'{done:,}/{total:,} 件' if total else f'{done:,} 件'
                rate = f'{speed:,.0f} 件/s' if speed else '-- 件/s'
            else:
                divisor, unit = (1024**3, 'GiB') if max(done, total) >= 1024**3 else (1024**2, 'MiB')
                amount = f'{done/divisor:.1f}/{total/divisor:.1f} {unit}' if total else f'{done/divisor:.1f} {unit}'
                rate = f'{speed/1024**2:.1f} MiB/s' if speed else '-- MiB/s'
            text += f' | {percent} | {amount} | {rate} | 残り {eta}'
        if self.notebook:
            from html import escape
            from IPython.display import HTML, display
            content = HTML('<pre>' + escape(text) + '</pre>')
            if self.display is None:
                self.display = display(content, display_id=True)
            else:
                self.display.update(content)
        elif self.stream.isatty():
            print('\r' + text + '\033[K', end='\n' if finished else '', file=self.stream, flush=True)
        else:
            print(text, file=self.stream, flush=True)


def _notify_progress(progress, **event):
    if progress is not None:
        progress(event)


def _file_progress(progress, index, completed, total_files):
    if progress is None:
        return None
    return lambda event: progress({**event, 'file_index': index, 'completed_files': completed,
                                   'file_total': total_files})


def destination(name):
    if name not in ARCHIVES + ANNOTATIONS:
        raise ValueError('Unexpected dataset filename')
    return storage_root() / 'raw' / ('archives' if name in ARCHIVES else 'annotations') / name


def read_odgt(path):
    """Never include raw annotations or IDs in exception messages."""
    runtime_path(path)
    with open(path, encoding='utf-8') as handle:
        for line_no, line in enumerate(handle, 1):
            try:
                record = json.loads(line)
            except (ValueError, TypeError):
                raise ValueError(f'Invalid ODGT record at line {line_no}: invalid JSON') from None
            if not isinstance(record, dict):
                raise ValueError(f'Invalid ODGT record at line {line_no}: expected a JSON object')
            image_id = record.get('ID')
            # Original CrowdHuman image stems contain a comma. Keep separators,
            # dots, whitespace and control characters excluded from filesystem IDs.
            if not isinstance(image_id, str) or not re.fullmatch(r'[A-Za-z0-9_,\-]+', image_id):
                raise ValueError(f'Invalid ODGT record at line {line_no}: invalid image ID')
            if not isinstance(record.get('gtboxes'), list):
                raise ValueError(f'Invalid ODGT record at line {line_no}: gtboxes must be a list')
            yield record


def validate_file(path, name, progress=None):
    runtime_path(path)
    if not Path(path).is_file():
        raise FileNotFoundError(f'Missing required file: {name}')
    if name in ARCHIVES:
        try:
            with zipfile.ZipFile(path) as z:
                entries = z.infolist()
                if not entries:
                    raise ValueError('Empty ZIP')
                if progress is None:
                    if z.testzip() is not None:
                        raise ValueError('ZIP CRC check failed')
                else:
                    total = sum(entry.file_size for entry in entries if not entry.is_dir())
                    processed = 0
                    _notify_progress(progress, stage='validation', filename=name, processed=0, total=total, unit='bytes')
                    for entry in entries:
                        if entry.is_dir():
                            continue
                        with z.open(entry) as handle:
                            while chunk := handle.read(1024**2):
                                processed += len(chunk)
                                _notify_progress(progress, stage='validation', filename=name,
                                                 processed=processed, total=total, unit='bytes')
                    _notify_progress(progress, stage='validation', filename=name, processed=processed,
                                     total=total, unit='bytes', finished=True)
        except zipfile.BadZipFile:
            raise ValueError(f'Invalid ZIP: {name}') from None
        return None
    count = 0
    expected = EXPECTED['train' if name == 'annotation_train.odgt' else 'val']
    _notify_progress(progress, stage='validation', filename=name, processed=0, total=expected, unit='records')
    for _ in read_odgt(path):
        count += 1
        if count % 100 == 0:
            _notify_progress(progress, stage='validation', filename=name, processed=count, total=expected, unit='records')
    if count == 0:
        raise ValueError('Empty annotation file')
    _notify_progress(progress, stage='validation', filename=name, processed=count, total=count, unit='records', finished=True)
    return count


def huggingface_links(repo_id=HF_REPO_ID, revision=HF_REVISION):
    """Pin all six original files to one public dataset commit, without Hub upload APIs."""
    if not isinstance(repo_id, str) or not re.fullmatch(r'[A-Za-z0-9_-]+/[A-Za-z0-9_.-]+', repo_id):
        raise ValueError('Invalid Hugging Face dataset repository ID')
    if not isinstance(revision, str) or not re.fullmatch(r'[a-f0-9]{40}', revision):
        raise ValueError('Use a full Hugging Face dataset commit SHA as revision')
    root = f'https://huggingface.co/datasets/{repo_id}/resolve/{revision}'
    return {name: f'{root}/{name}?download=true' for name in ARCHIVES + ANNOTATIONS}


def _safe_download_message(message):
    """Never retain signed URLs, credentials, response bodies or annotation content."""
    text = re.sub(r'https?://[^\s\"\'<>]+', '[URL removed]', str(message))
    text = re.sub(r'(?i)\b(?:hf_[A-Za-z0-9]+|Bearer\s+[^\s,;]+)', '[credential removed]', text)
    text = re.sub(r'(?i)((?:token|password|authorization|signature|credential|api[_-]?key)\s*[:=]\s*)[^\s,;]+', r'\1[redacted]', text)
    return text[:1000]


def _download_event(log_path, context, event, **details):
    path = runtime_path(log_path)
    runtime_path(path.parent).mkdir(parents=True, exist_ok=True)
    entry = {'timestamp_utc': datetime.now(timezone.utc).isoformat(),
             'event': event, **context, **details}
    with open(path, 'a', encoding='utf-8') as handle:
        handle.write(json.dumps(entry, ensure_ascii=False) + '\n')


def _download_error_details(error):
    chain, seen = [], set()
    current = error
    while current is not None and id(current) not in seen:
        seen.add(id(current))
        item = {'type': type(current).__name__, 'message': _safe_download_message(current)}
        if isinstance(current, urllib.error.HTTPError):
            item['http_status'] = current.code
            item['reason'] = _safe_download_message(current.reason)
            if current.headers:
                item['response_headers'] = {
                    name: _safe_download_message(current.headers[name])
                    for name in ('X-Request-Id', 'Retry-After') if name in current.headers}
        elif isinstance(current, urllib.error.URLError):
            item['reason'] = _safe_download_message(current.reason)
        if isinstance(current, OSError) and current.errno is not None:
            item['errno'] = current.errno
        # Stack locations only: no source lines, locals, response payloads or data IDs.
        item['stack'] = [{'file': Path(frame.filename).name, 'line': frame.lineno,
                          'function': frame.name}
                         for frame in traceback.extract_tb(current.__traceback__)]
        chain.append(item)
        current = current.__cause__ if current.__cause__ is not None else current.__context__
    return chain


def download_huggingface(repo_id=HF_REPO_ID, revision=HF_REVISION, progress=None):
    """Download public files and retain sanitized diagnostics inside the runtime."""
    log_path = runtime_path(storage_root() / 'logs/download.jsonl')
    context = {'attempt_id': uuid.uuid4().hex, 'stage': 'configuration'}
    temp = None
    completed = 0
    total_files = len(ARCHIVES + ANNOTATIONS)
    try:
        links = huggingface_links(repo_id, revision)
        context.update(repo_id=repo_id, revision=revision)
        _download_event(log_path, context, 'start')
        needed = []
        for index, name in enumerate(ARCHIVES + ANNOTATIONS, 1):
            context.update(filename=name, stage='existing_file_validation')
            path = runtime_path(destination(name))
            if path.exists():
                validate_file(path, name, progress=_file_progress(progress, index, completed, total_files))
                completed += 1
                _notify_progress(progress, stage='reused', filename=name, completed_files=completed, file_total=total_files)
                _download_event(log_path, context, 'reused', bytes_on_disk=path.stat().st_size)
            else:
                needed.append(name)
        for name in needed:
            context = {'attempt_id': context['attempt_id'], 'repo_id': repo_id,
                       'revision': revision, 'filename': name, 'stage': 'http_request',
                       'bytes_received': 0}
            path = runtime_path(destination(name))
            temp = runtime_path(path.with_suffix(path.suffix + '.part'))
            index = (ARCHIVES + ANNOTATIONS).index(name) + 1
            current_progress = _file_progress(progress, index, completed, total_files)
            _notify_progress(current_progress, stage='request', filename=name)
            _download_event(log_path, context, 'file_start')
            # No auth headers, cookies, response bodies or signed redirect URLs are logged.
            with urllib.request.urlopen(links[name], timeout=120) as response, open(temp, 'wb') as out:
                context['http_status'] = getattr(response, 'status', None)
                context['response_host'] = urlsplit(response.geturl()).hostname if hasattr(response, 'geturl') else 'huggingface.co'
                content_type = response.headers.get('Content-Type', '')
                context['content_type'] = _safe_download_message(content_type)
                if 'text/html' in content_type.lower():
                    raise RuntimeError('Hugging Face returned an access page instead of a file')
                length = int(response.headers.get('Content-Length', 0))
                context.update(expected_bytes=length, stage='disk_space_check',
                               free_bytes=shutil.disk_usage(storage_root()).free)
                _download_event(log_path, context, 'response')
                if length and context['free_bytes'] < length + 2 * 1024**3:
                    raise OSError('Insufficient download disk space')
                context['stage'] = 'transfer'
                last_progress = 0
                _notify_progress(current_progress, stage='transfer', filename=name, processed=0, total=length, unit='bytes')
                while chunk := response.read(1024**2):
                    context['free_bytes'] = shutil.disk_usage(storage_root()).free
                    if context['free_bytes'] < len(chunk) + 1024**3:
                        raise OSError('Insufficient download disk space')
                    out.write(chunk)
                    context['bytes_received'] += len(chunk)
                    _notify_progress(current_progress, stage='transfer', filename=name,
                                     processed=context['bytes_received'], total=length, unit='bytes')
                    if context['bytes_received'] - last_progress >= 64 * 1024**2:
                        _download_event(log_path, context, 'progress')
                        last_progress = context['bytes_received']
                if length and context['bytes_received'] != length:
                    raise ValueError('Incomplete Hugging Face response')
            _notify_progress(current_progress, stage='transfer', filename=name,
                             processed=context['bytes_received'], total=length, unit='bytes', finished=True)
            context['stage'] = 'downloaded_file_validation'
            validate_file(temp, name, progress=current_progress)
            temp.replace(path)
            temp = None
            completed += 1
            _notify_progress(progress, stage='file_complete', filename=name, completed_files=completed, file_total=total_files)
            _download_event(log_path, context, 'file_complete')
        context['stage'] = 'save_source_metadata'
        source = runtime_path(storage_root() / 'raw/download_source.json')
        source.write_text(json.dumps({'requested_repo_id': repo_id, 'requested_revision': revision,
                                      'downloaded': needed,
                                      'reused_existing': [name for name in ARCHIVES + ANNOTATIONS if name not in needed]}, indent=2))
        _download_event(log_path, context, 'complete', downloaded_files=len(needed),
                        reused_files=len(ARCHIVES + ANNOTATIONS) - len(needed))
        _notify_progress(progress, stage='complete', completed_files=completed, file_total=total_files)
        return log_path
    except (Exception, KeyboardInterrupt) as error:
        cleanup_error = None
        if temp is not None:
            try:
                temp.unlink(missing_ok=True)
            except OSError as cleanup:
                cleanup_error = _safe_download_message(cleanup)
        log_note = f'Log: {log_path}'
        try:
            _download_event(log_path, context, 'interrupted' if isinstance(error, KeyboardInterrupt) else 'error',
                            errors=_download_error_details(error), cleanup_error=cleanup_error)
        except OSError as log_error:
            log_note = f'Could not write log {log_path}: {_safe_download_message(log_error)}'
        _notify_progress(progress, stage='interrupted' if isinstance(error, KeyboardInterrupt) else 'error',
                         filename=context.get('filename', ''), completed_files=completed, file_total=total_files)
        if isinstance(error, KeyboardInterrupt):
            raise
        raise RuntimeError(f'Hugging Face download failed: {context.get("filename", "configuration")}; '
                           f'{type(error).__name__}: {_safe_download_message(error)}. '
                           + log_note) from None


def upload_missing():
    """Colab upload widget writes files directly to raw; it never prints file contents."""
    from google.colab import files
    import os
    previous = Path.cwd()
    try:
        os.chdir(runtime_path(storage_root() / 'raw'))
        uploaded = files.upload()
        del uploaded  # Browser upload may use substantial RAM for large archives.
        for name in ARCHIVES + ANNOTATIONS:
            staged = storage_root() / 'raw' / name
            if staged.exists():
                target = destination(name)
                if target.exists():
                    validate_file(target, name)
                    staged.unlink()
                else:
                    validate_file(staged, name)
                    staged.replace(target)
    finally:
        os.chdir(previous)


def inventory(progress=None):
    counts = {}
    files = ARCHIVES + ANNOTATIONS
    for index, name in enumerate(files, 1):
        count = validate_file(destination(name), name, progress=_file_progress(progress, index, index - 1, len(files)))
        if count is not None:
            counts[name] = count
    for split in EXPECTED:
        if counts[f'annotation_{split}.odgt'] != EXPECTED[split]:
            raise ValueError('Official annotation count mismatch')
    _notify_progress(progress, stage='complete', completed_files=len(files), file_total=len(files))
    return {'annotation_counts': counts, 'free_gib': round(shutil.disk_usage(storage_root()).free / 1024**3, 2)}


def extract_archives():
    root = runtime_path(storage_root() / 'raw/images')
    entries = []
    seen = set()
    for name in ARCHIVES:
        archive = destination(name)
        validate_file(archive, name)
        with zipfile.ZipFile(archive) as z:
            for item in z.infolist():
                if item.is_dir():
                    continue
                parts = Path(item.filename).parts
                if len(parts) != 2 or parts[0] != 'Images' or Path(parts[1]).suffix.lower() not in ('.jpg', '.jpeg'):
                    raise ValueError('Unexpected archive member')
                if stat.S_ISLNK(item.external_attr >> 16) or parts[1] in seen:
                    raise ValueError('Duplicate/unsafe archive member')
                seen.add(parts[1])
                target = runtime_path(root / parts[1])
                entries.append((archive, item.filename, target, item.file_size, item.CRC))
    required = sum(size for _, _, p, size, _ in entries if not p.exists() or p.stat().st_size != size)
    if shutil.disk_usage(storage_root()).free < required + 2 * 1024**3:
        raise OSError('Insufficient extraction space (plus 2 GiB reserve)')
    # Compare CRCs before reuse; a same-size corrupt image must also be replaced.
    for archive in map(destination, ARCHIVES):
        with zipfile.ZipFile(archive) as z:
            for source_archive, member, target, size, crc in entries:
                if source_archive != archive:
                    continue
                if target.exists() and target.stat().st_size == size:
                    actual = 0
                    with open(target, 'rb') as existing:
                        for chunk in iter(lambda: existing.read(1024**2), b''):
                            actual = zlib.crc32(chunk, actual)
                    if actual == crc:
                        continue
                part = target.with_suffix('.part')
                with z.open(member) as source, open(part, 'wb') as out:
                    shutil.copyfileobj(source, out)
                part.replace(target)
    return {'archive_images': len(entries)}


def image_size(path):
    runtime_path(path)
    from PIL import Image
    try:
        with Image.open(path) as image:
            size = image.size
            image.verify()
        if min(size) <= 0:
            raise ValueError()
        return size
    except Exception:
        raise ValueError('Missing or corrupt dataset image') from None


def inspect_dataset():
    """Check every image, including DEBUG runs; reject split overlap and duplicates."""
    splits = {}
    sizes = {}
    for split in ('train', 'val'):
        ids = set()
        for record in read_odgt(destination(f'annotation_{split}.odgt')):
            key = record['ID']
            if key in ids:
                raise ValueError('Duplicate image ID within split')
            ids.add(key)
            sizes[key] = image_size(storage_root() / 'raw/images' / f'{key}.jpg')
        splits[split] = ids
    if splits['train'] & splits['val']:
        raise ValueError('Train/val overlap detected')
    return sizes, {split: len(ids) for split, ids in splits.items()}


def _copy_checksum(path, name, progress=None):
    """Read only an explicitly approved source/cache file; never expose its contents."""
    import hashlib
    digest = hashlib.sha256()
    total = path.stat().st_size
    processed = 0
    _notify_progress(progress, stage='checksum', filename=name, processed=0, total=total, unit='bytes')
    with open(path, 'rb') as handle:
        while chunk := handle.read(1024**2):
            digest.update(chunk)
            processed += len(chunk)
            _notify_progress(progress, stage='checksum', filename=name, processed=processed, total=total, unit='bytes')
    _notify_progress(progress, stage='checksum', filename=name, processed=processed, total=total, unit='bytes', finished=True)
    return digest.hexdigest()


def copy_drive_dataset(source_dir, progress=None):
    """Copy six private Drive cache files into the active runtime before processing.

    Drive layout matches raw/archives and raw/annotations. Sources are read only;
    all labels, images, training runs and temporary copy files stay in the runtime.
    """
    import hashlib
    source_root = Path(source_dir).expanduser().absolute()
    if source_root.resolve() != source_root:
        raise ValueError('Drive dataset directory must not traverse symlinks')
    if source_root == storage_root():
        raise ValueError('Drive source and runtime destination must differ')
    files = ARCHIVES + ANNOTATIONS
    log_path = runtime_path(storage_root() / 'logs/drive_copy.jsonl')
    context = {'attempt_id': uuid.uuid4().hex, 'stage': 'source_inventory'}
    temporary = None
    completed = 0
    try:
        sources = {}
        for name in files:
            relative = Path('raw') / ('archives' if name in ARCHIVES else 'annotations') / name
            source = source_root / relative
            if source_root not in source.resolve().parents:
                raise ValueError('Drive source file escapes its dataset directory')
            if not source.is_file():
                raise FileNotFoundError(f'Missing Drive cache file: {name}')
            sources[name] = source
        required = sum(source.stat().st_size for name, source in sources.items() if not destination(name).exists())
        if shutil.disk_usage(storage_root()).free < required + 2 * 1024**3:
            raise OSError('Insufficient runtime space for Drive copy (plus 2 GiB reserve)')
        _download_event(log_path, context, 'start', file_total=len(files), required_bytes=required)
        records = []
        for index, name in enumerate(files, 1):
            source = sources[name]
            target = runtime_path(destination(name))
            runtime_path(target.parent).mkdir(parents=True, exist_ok=True)
            current_progress = _file_progress(progress, index, completed, len(files))
            context.update(stage='copy', filename=name, bytes_received=0, expected_bytes=source.stat().st_size)
            if target.exists():
                context['stage'] = 'existing_file_checksum'
                source_hash = _copy_checksum(source, name, current_progress)
                if _copy_checksum(target, name, current_progress) != source_hash:
                    raise ValueError(f'Runtime file differs from Drive cache: {name}; remove the local file explicitly before retrying')
                validate_file(target, name, progress=current_progress)
                action = 'reused'
            else:
                temporary = runtime_path(target.with_suffix(target.suffix + '.copying'))
                digest = hashlib.sha256()
                _notify_progress(current_progress, stage='copy', filename=name, processed=0,
                                 total=context['expected_bytes'], unit='bytes')
                with open(source, 'rb') as incoming, open(temporary, 'wb') as outgoing:
                    while chunk := incoming.read(1024**2):
                        if shutil.disk_usage(storage_root()).free < len(chunk) + 1024**3:
                            raise OSError('Insufficient runtime space during Drive copy')
                        outgoing.write(chunk)
                        digest.update(chunk)
                        context['bytes_received'] += len(chunk)
                        _notify_progress(current_progress, stage='copy', filename=name,
                                         processed=context['bytes_received'], total=context['expected_bytes'], unit='bytes')
                if context['bytes_received'] != context['expected_bytes']:
                    raise ValueError('Drive source changed or copy was incomplete')
                _notify_progress(current_progress, stage='copy', filename=name,
                                 processed=context['bytes_received'], total=context['expected_bytes'], unit='bytes', finished=True)
                source_hash = digest.hexdigest()
                context['stage'] = 'copied_file_checksum'
                if _copy_checksum(temporary, name, current_progress) != source_hash:
                    raise ValueError('Drive copy checksum mismatch')
                context['stage'] = 'copied_file_validation'
                validate_file(temporary, name, progress=current_progress)
                temporary.replace(target)
                temporary = None
                action = 'copied'
            completed += 1
            records.append({'filename': name, 'sha256': source_hash, 'action': action})
            _download_event(log_path, context, action, sha256=source_hash, completed_files=completed)
            _notify_progress(progress, stage='reused' if action == 'reused' else 'file_complete',
                             filename=name, completed_files=completed, file_total=len(files))
        manifest = runtime_path(storage_root() / 'raw/drive_copy_source.json')
        manifest.write_text(json.dumps({'source_directory': str(source_root), 'files': records}, indent=2))
        context['stage'] = 'complete'
        _download_event(log_path, context, 'complete', completed_files=completed)
        _notify_progress(progress, stage='complete', completed_files=completed, file_total=len(files))
        return log_path
    except (Exception, KeyboardInterrupt) as error:
        if temporary is not None:
            temporary.unlink(missing_ok=True)
        _download_event(log_path, context, 'interrupted' if isinstance(error, KeyboardInterrupt) else 'error',
                        errors=_download_error_details(error))
        _notify_progress(progress, stage='interrupted' if isinstance(error, KeyboardInterrupt) else 'error',
                         filename=context.get('filename', ''), completed_files=completed, file_total=len(files))
        if isinstance(error, KeyboardInterrupt):
            raise
        raise RuntimeError(f'Drive copy failed: {_safe_download_message(error)}. Log: {log_path}') from None

def mount_dataset_drive(mount_point):
    from google.colab import drive
    drive.mount(mount_point)

if not TERMS_REVIEWED:
    raise RuntimeError('利用条件とGoogle Drive・Colabでの保存・処理可否を確認し、TERMS_REVIEWED=Trueにしてください。')
mount_dataset_drive(DRIVE_MOUNT_POINT)
drive_dataset = Path(DRIVE_DATASET_DIR)
mount_root = Path(DRIVE_MOUNT_POINT).absolute()
if mount_root not in drive_dataset.absolute().parents:
    raise ValueError('DRIVE_DATASET_DIRはマウントしたGoogle Drive配下を指定してください。')
if DOWNLOAD_MISSING_TO_DRIVE:
    with local_download_storage(drive_dataset):
        for part in ('raw/archives', 'raw/annotations', 'logs'):
            runtime_path(drive_dataset / part).mkdir(parents=True, exist_ok=True)
        drive_log = download_huggingface(HF_DATASET_REPO, HF_DATASET_REVISION,
                                         progress=DownloadProgress(notebook=True))
        print('Driveへの保管完了。取得ログ:', drive_log)
copy_log = copy_drive_dataset(drive_dataset, progress=DownloadProgress(notebook=True))
print('Colabへのコピー・検証完了。ログ:', copy_log)

## 7. ファイル整合性・空き容量確認
必須6ファイル、全ZIPのCRC、ODGT構造とCrowdHumanの件数(train 15,000 / val 4,370)を確認します。
データの内容や画像IDは表示せず件数だけを出力します。展開前に非圧縮サイズと空き容量も確認します。

In [ ]:
print(inventory(progress=DownloadProgress(notebook=True)))

## 8. ZIPの安全な解凍
raw/imagesに1組だけ展開します。想定外のパス・シンボリックリンク・重複画像を拒否します。
既存画像はCRCが一致すれば再展開しません。画像・アノテーションはすべてランタイム内に留まります。

In [ ]:
print(extract_archives())

## 9. ODGTの解析と画像の検証
1行ずつJSONとして読み込み、全画像の存在・画像ヘッダの整合性・サイズを調べます。
同一split内の重複ID、train/val間のID重複はエラーにします。DEBUGでも入力全体の分離を確認します。

In [ ]:
image_sizes, split_counts = inspect_dataset()
print({'validated_images': split_counts})

## 10. YOLO形式への変換
person以外を除外し、vboxはextra.ignore、hboxはhead_attr.ignoreを独立に判定します。
標準YOLOはignore領域を表現できません。除外領域の検出は背景に対する誤検出として扱われる可能性があり、CrowdHuman公式評価とは異なります。
bboxをクリッピング・正規化し、不正データ件数だけ記録します。画像はシンボリックリンクで共用します。
DEBUGと本番で同じprocessedを使い回すことは拒否します。切替時は下の手順に従いprocessedを明示的に削除してください。

In [ ]:
"""Separate body/head ignore policy and streaming YOLO label conversion."""
import hashlib
import json
import math
import shutil
from collections import Counter
from pathlib import Path

NAMES = {'vbox': {0: 'person'}, 'hbox': {0: 'head'}, 'both': {0: 'person', 1: 'head'}}


def normalize_box(box, width, height):
    if not isinstance(box, (list, tuple)) or len(box) != 4:
        raise ValueError('Invalid bbox')
    if any(isinstance(v, bool) or not isinstance(v, (int, float)) or not math.isfinite(v) for v in box):
        raise ValueError('Invalid bbox')
    x, y, w, h = box
    if width <= 0 or height <= 0 or w <= 0 or h <= 0:
        raise ValueError('Nonpositive dimensions')
    x1, y1 = max(0, min(width, x)), max(0, min(height, y))
    x2, y2 = max(0, min(width, x + w)), max(0, min(height, y + h))
    if x2 <= x1 or y2 <= y1:
        raise ValueError('Empty clipped bbox')
    return ((x1 + x2) / (2 * width), (y1 + y2) / (2 * height), (x2 - x1) / width, (y2 - y1) / height)


def ignored(annotation, attr):
    value = annotation.get(attr, {})
    if not isinstance(value, dict) or value.get('ignore', 0) not in (0, 1):
        raise ValueError('Invalid ignore attribute')
    return value.get('ignore', 0) == 1


def convert_boxes(boxes, mode, width, height):
    if mode not in NAMES:
        raise ValueError('Unsupported mode')
    labels, stats = [], Counter()
    fields = ('vbox', 'hbox') if mode == 'both' else (mode,)
    for annotation in boxes:
        if not isinstance(annotation, dict):
            stats['invalid_annotation'] += 1
            continue
        if annotation.get('tag') != 'person':
            stats['non_person'] += 1
            continue
        for field in fields:
            try:
                if ignored(annotation, 'extra' if field == 'vbox' else 'head_attr'):
                    stats[f'{field}_ignored'] += 1
                    continue
                box = annotation.get(field)
                normalized = normalize_box(box, width, height)
                class_id = 1 if mode == 'both' and field == 'hbox' else 0
                labels.append((class_id, *normalized))
                stats[f'{field}_accepted'] += 1
                x, y, w, h = box
                if x < 0 or y < 0 or x + w > width or y + h > height:
                    stats[f'{field}_clipped'] += 1
            except (ValueError, TypeError):
                stats[f'{field}_invalid'] += 1
    return labels, stats


def fingerprint(sizes, limit):
    digest = hashlib.sha256()
    for split in ('train', 'val'):
        with open(destination(f'annotation_{split}.odgt'), 'rb') as handle:
            for chunk in iter(lambda: handle.read(1024**2), b''):
                digest.update(chunk)
    digest.update(json.dumps(sorted(sizes.items())).encode())
    digest.update(str(limit).encode())
    return digest.hexdigest()


def convert_dataset(mode, sizes, limit=None):
    if mode not in NAMES or (limit is not None and limit <= 0):
        raise ValueError('Invalid conversion configuration')
    root = runtime_path(BASE / 'processed' / mode)
    signature = fingerprint(sizes, limit)
    manifest = root / 'conversion.json'
    if manifest.exists():
        previous = json.loads(manifest.read_text())
        if previous['fingerprint'] == signature:
            validate_labels(root, mode)
            return root / 'data.yaml'
        raise ValueError('Dataset changed; remove processed mode and matching run explicitly')
    staging = root.with_name(mode + '.building')
    if staging.exists():
        shutil.rmtree(runtime_path(staging))
    counts = Counter()
    selected = set()
    try:
        for split in ('train', 'val'):
            images = staging / 'images' / split
            labels_dir = staging / 'labels' / split
            images.mkdir(parents=True)
            labels_dir.mkdir(parents=True)
            for index, record in enumerate(read_odgt(destination(f'annotation_{split}.odgt'))):
                if limit is not None and index >= limit:
                    break
                key = record['ID']
                if key in selected:
                    raise ValueError('Duplicate image or split overlap')
                selected.add(key)
                width, height = sizes[key]
                source = runtime_path(BASE / 'raw/images' / f'{key}.jpg')
                if not source.is_file():
                    raise FileNotFoundError('Missing source image')
                rows, stats = convert_boxes(record['gtboxes'], mode, width, height)
                counts.update({f'{split}_{k}': v for k, v in stats.items()})
                counts[f'{split}_images'] += 1
                (images / source.name).symlink_to(source)
                (labels_dir / f'{key}.txt').write_text(''.join(' '.join([str(row[0]), *(f'{v:.10f}' for v in row[1:])]) + '\n' for row in rows))
        if not counts['train_images'] or not counts['val_images']:
            raise ValueError('Both splits must be nonempty')
        (staging / 'conversion.json').write_text(json.dumps({'fingerprint': signature, 'limit': limit, 'counts': counts}, indent=2))
        if root.exists():
            raise ValueError('Incomplete processed directory; remove it explicitly')
        staging.rename(root)
        write_yaml(root, mode)
        validate_labels(root, mode)
    except Exception:
        if staging.exists():
            shutil.rmtree(staging)
        raise
    return root / 'data.yaml'


def write_yaml(root, mode):
    import yaml
    root = runtime_path(root)
    data = {'path': str(root), 'train': 'images/train', 'val': 'images/val', 'names': NAMES[mode]}
    (root / 'data.yaml').write_text(yaml.safe_dump(data, sort_keys=False))


def validate_labels(root, mode):
    import yaml
    root = runtime_path(root)
    data = yaml.safe_load((root / 'data.yaml').read_text())
    if data != {'path': str(root), 'train': 'images/train', 'val': 'images/val', 'names': NAMES[mode]}:
        raise ValueError('data.yaml mismatch')
    ids = []
    totals = {}
    for split in ('train', 'val'):
        images = list((root / 'images' / split).glob('*.jpg'))
        label_files = list((root / 'labels' / split).glob('*.txt'))
        if not images or {p.stem for p in images} != {p.stem for p in label_files}:
            raise ValueError('Images/labels mismatch')
        boxes = 0
        ids.append({p.stem for p in images})
        for image in images:
            runtime_path(image)
            if not image.is_file():
                raise FileNotFoundError('Broken image symlink')
        for path in label_files:
            runtime_path(path)
            for line in path.read_text().splitlines():
                fields = line.split()
                if len(fields) != 5 or fields[0] not in map(str, NAMES[mode]):
                    raise ValueError('Invalid class or label columns')
                values = list(map(float, fields[1:]))
                x, y, w, h = values
                if not all(math.isfinite(v) and 0 <= v <= 1 for v in values) or min(w, h) <= 0:
                    raise ValueError('Invalid normalized label')
                if x-w/2 < -1e-8 or x+w/2 > 1+1e-8 or y-h/2 < -1e-8 or y+h/2 > 1+1e-8:
                    raise ValueError('Label crosses image boundary')
                boxes += 1
        if not boxes:
            raise ValueError('Split has no usable labels')
        totals[split] = {'images': len(images), 'boxes': boxes}
    if ids[0] & ids[1]:
        raise ValueError('Split overlap')
    return totals

yaml_paths = {}
for mode in MODES_TO_RUN:
    yaml_paths[mode] = convert_dataset(mode, image_sizes, DEBUG_LIMIT if DEBUG_MODE else None)
    report = __import__('json').loads((yaml_paths[mode].parent / 'conversion.json').read_text())
    print(mode, report['counts'])

## 11. 合成テストとアノテーションの検証
合成画像・合成ODGTで変換処理をテストします。実画像の表示は行いません。
クラスID・正規化座標・画像とラベルの対応・split分離を検証します。空ラベル画像は背景画像として残します。

In [ ]:
# この合成テストもノートブック内で直接実行します。実データは使いません。
import io
import tempfile
import unittest
from unittest.mock import patch
from PIL import Image


class NotebookSyntheticTests(unittest.TestCase):
    def setUp(self):
        self.temp = tempfile.TemporaryDirectory(prefix='synthetic-notebook-')
        self.addCleanup(self.temp.cleanup)
        self.root = Path(self.temp.name)
        # 合成データに限り、一時ディレクトリを使って保存先を差し替えます。
        self.guard = patch.dict(globals(), BASE=self.root)
        self.guard.start()
        self.addCleanup(self.guard.stop)
        for relative in ('raw/annotations', 'raw/images', 'processed'):
            (self.root / relative).mkdir(parents=True)
        self.person = {'tag': 'person', 'vbox': [10, 20, 30, 40], 'hbox': [20, 20, 10, 10]}
        for split in ('train', 'val'):
            key = f'synthetic,{split}'
            Image.new('RGB', (100, 100)).save(self.root / 'raw/images' / f'{key}.jpg')
            record = {'ID': key, 'gtboxes': [self.person]}
            destination(f'annotation_{split}.odgt').write_text(json.dumps(record) + '\n')

    def test_vbox(self):
        rows, _ = convert_boxes([self.person], 'vbox', 100, 100)
        self.assertEqual(rows, [(0, .25, .4, .3, .4)])

    def test_hbox(self):
        rows, _ = convert_boxes([self.person], 'hbox', 100, 100)
        self.assertEqual(rows, [(0, .25, .25, .1, .1)])

    def test_independent_ignore(self):
        for body, head, expected in ((0, 0, [0, 1]), (1, 0, [1]), (0, 1, [0]), (1, 1, [])):
            annotation = dict(self.person, extra={'ignore': body}, head_attr={'ignore': head})
            rows, _ = convert_boxes([annotation], 'both', 100, 100)
            self.assertEqual([row[0] for row in rows], expected)

    def test_clipping(self):
        self.assertEqual(normalize_box([-20, -10, 150, 120], 100, 100), (.5, .5, 1, 1))

    def test_invalid_boxes(self):
        for box in (None, [], [0, 0, 0, 1], [0, 0, -1, 1], [101, 0, 5, 5], [0, 0, float('nan'), 1]):
            rows, stats = convert_boxes([dict(self.person, vbox=box)], 'vbox', 100, 100)
            self.assertEqual(rows, [])
            self.assertEqual(stats['vbox_invalid'], 1)

    def test_non_person(self):
        rows, _ = convert_boxes([dict(self.person, tag='mask')], 'both', 100, 100)
        self.assertEqual(rows, [])

    def test_class_ids(self):
        rows, _ = convert_boxes([self.person], 'both', 100, 100)
        self.assertEqual([row[0] for row in rows], [0, 1])
        self.assertEqual(NAMES['both'], {0: 'person', 1: 'head'})

    def test_split_separation(self):
        sizes, counts = inspect_dataset()
        root = convert_dataset('vbox', sizes).parent
        self.assertEqual(counts, {'train': 1, 'val': 1})
        self.assertTrue((root / 'images/train/synthetic,train.jpg').is_symlink())
        self.assertFalse((root / 'images/val/synthetic,train.jpg').exists())
        self.assertTrue((root / 'labels/val/synthetic,val.txt').exists())

    def test_yaml(self):
        import yaml
        sizes, _ = inspect_dataset()
        for mode in ('vbox', 'hbox', 'both'):
            path = convert_dataset(mode, sizes)
            data = yaml.safe_load(path.read_text())
            self.assertEqual(data['path'], str(path.parent))
            self.assertEqual(data['train'], 'images/train')
            self.assertEqual(data['val'], 'images/val')
            self.assertEqual(data['names'], NAMES[mode])
            self.assertEqual(validate_labels(path.parent, mode)['train']['images'], 1)

    def test_ignore_rules(self):
        with tempfile.TemporaryDirectory(prefix='synthetic-git-policy-') as directory:
            root = Path(directory)
            (root / '.gitignore').write_text(GITIGNORE_CONTENT)
            subprocess.run(['git', 'init', '-q', str(root)], check=True)
            for name in ('CrowdHuman/a', 'crowdhuman/a', 'datasets/a', 'data/a', 'a.odgt', 'a.zip',
                         'a.jpg', 'a.jpeg', 'a.png', 'a.webp', 'runs/a', 'weights/a', 'checkpoints/a',
                         'a.pt', 'a.pth', 'a.onnx', '.env', '.env.local', 'a.key', '__pycache__/a', '.ipynb_checkpoints/a'):
                self.assertEqual(subprocess.run(['git', '-C', str(root), 'check-ignore', '-q', '--', name]).returncode, 0)

    def test_missing_file(self):
        with self.assertRaises(FileNotFoundError):
            list(read_odgt(self.root / 'raw/annotations/missing.odgt'))
        with self.assertRaises(ValueError):
            image_size(self.root / 'raw/images/missing.jpg')

    def test_storage_boundary(self):
        with self.assertRaises(ValueError):
            runtime_path(self.root.parent / 'outside')


suite = unittest.defaultTestLoader.loadTestsFromTestCase(NotebookSyntheticTests)
with local_console(BASE / 'synthetic_tests.log'):
    result = unittest.TextTestRunner(verbosity=2).run(suite)
if not result.wasSuccessful():
    raise RuntimeError('合成テスト失敗。synthetic_tests.logを確認してください。')
print({'synthetic_tests': result.testsRun, 'status': 'passed'})
for mode in MODES_TO_RUN:
    print(mode, validate_labels(yaml_paths[mode].parent, mode))

## 12. data.yamlの確認とGoogle Driveへの進捗保存
変換時にdata.yamlを生成済みです。各モードにimages/train・images/val・labels/train・labels/valがあります。
vboxは0:person、hboxは0:head、bothは0:person / 1:headです。データの内容は表示しません。
検証後、展開画像・ODGT・変換済みラベルとYAMLをDRIVE_PREPARED_DIRへ非公開保存します。容量と転送時間が必要です。
保存完了の表示を確認してからランタイムを終了してください。
新しいランタイムではRESTORE_PREPARED_FROM_DRIVE=Trueにして手順2〜5を実行し、6〜12を飛ばして13へ進みます。

In [ ]:
"""Save prepared private data once; restore without downloading or converting."""
import hashlib
import json
import os
import shutil
import tarfile
import tempfile
import uuid
from pathlib import Path, PurePosixPath


def _prepared_hash(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(8 * 1024**2), b''):
            digest.update(chunk)
    return digest.hexdigest()


def _prepared_root(directory):
    root = Path(directory).absolute()
    if root.resolve() != root:
        raise ValueError('Prepared data directory must not traverse symlinks')
    return root


def _prepared_parts(settings):
    modes = settings['modes']
    if not modes or len(set(modes)) != len(modes) or any(m not in ('vbox', 'hbox', 'both') for m in modes):
        raise ValueError('Invalid prepared modes')
    return ['raw/images', 'raw/annotations'] + ['processed/' + mode for mode in modes]


def save_prepared_data(directory, settings):
    """Publish latest.json only after a complete archive has been verified."""
    root = _prepared_root(directory)
    parts = _prepared_parts(settings)
    root.mkdir(parents=True, exist_ok=True)
    name = 'prepared_' + uuid.uuid4().hex + '.tar'
    archive = root / name
    temporary = root / (name + '.incomplete')
    try:
        with tarfile.open(temporary, 'w') as tar:
            for part in parts:
                source = runtime_path(BASE / part)
                if not source.is_dir():
                    raise FileNotFoundError('Prepared data is missing: ' + part)
                for path in [source, *sorted(source.rglob('*'))]:
                    relative = path.relative_to(BASE).as_posix()
                    runtime_path(path)
                    info = tar.gettarinfo(str(path), arcname=relative)
                    if info.issym():
                        target = path.resolve()
                        image_root = BASE / 'raw/images'
                        if image_root not in target.parents or not target.is_file():
                            raise ValueError('Prepared link must point to an image')
                        info.linkname = os.path.relpath(target, path.parent)
                        tar.addfile(info)
                    elif info.isfile():
                        with open(path, 'rb') as handle:
                            tar.addfile(info, handle)
                    elif info.isdir():
                        tar.addfile(info)
                    else:
                        raise ValueError('Unsupported prepared file')
        checksum = _prepared_hash(temporary)
        temporary.rename(archive)
        manifest = {'schema': 1, 'archive': name, 'sha256': checksum,
                    'size': archive.stat().st_size, 'settings': settings}
        pointer = root / ('latest.' + uuid.uuid4().hex + '.tmp')
        pointer.write_text(json.dumps(manifest, ensure_ascii=False, indent=2))
        pointer.replace(root / 'latest.json')
        return archive
    finally:
        temporary.unlink(missing_ok=True)


def restore_prepared_data(directory, settings):
    """Verify paths and checksum, extract in staging, then install into empty dirs."""
    root = _prepared_root(directory)
    parts = _prepared_parts(settings)
    manifest = json.loads((root / 'latest.json').read_text())
    if manifest.get('schema') != 1 or manifest.get('settings') != settings:
        raise ValueError('Prepared data settings mismatch; use the saved modes/DEBUG settings')
    name = manifest['archive']
    if not isinstance(name, str) or not name.startswith('prepared_') or not name.endswith('.tar') or Path(name).name != name:
        raise ValueError('Invalid prepared archive name')
    archive = _prepared_root(root / name)
    if archive.stat().st_size != manifest['size'] or _prepared_hash(archive) != manifest['sha256']:
        raise ValueError('Prepared archive checksum mismatch')
    for part in parts:
        target = runtime_path(BASE / part)
        if target.exists() and (not target.is_dir() or any(target.iterdir())):
            raise FileExistsError('Restore requires an empty local directory: ' + part)
    with tarfile.open(archive, 'r:') as tar:
        members = tar.getmembers()
        seen, links = set(), set()
        for member in members:
            path = PurePosixPath(member.name)
            if path.is_absolute() or '..' in path.parts or member.name in seen:
                raise ValueError('Unsafe prepared archive path')
            if not any(member.name == part or member.name.startswith(part + '/') for part in parts):
                raise ValueError('Unexpected prepared archive path')
            seen.add(member.name)
            if member.issym():
                target = (BASE / member.name).parent / member.linkname
                resolved = target.resolve()
                if (BASE / 'raw/images') not in resolved.parents or Path(member.linkname).is_absolute():
                    raise ValueError('Unsafe prepared archive link')
                links.add(member.name)
            elif not (member.isfile() or member.isdir()):
                raise ValueError('Unsupported prepared archive entry')
        for member in members:
            if any(parent.as_posix() in links for parent in PurePosixPath(member.name).parents):
                raise ValueError('Prepared archive traverses a link')
        required = sum(member.size for member in members if member.isfile())
        if shutil.disk_usage(BASE).free < required + 1024**2:
            raise OSError('Insufficient local space for prepared data')
        with tempfile.TemporaryDirectory(prefix='.prepared-', dir=BASE) as temporary:
            stage = Path(temporary)
            # Extract regular entries first; links are created only after all writes.
            for member in members:
                target = stage / member.name
                target.parent.mkdir(parents=True, exist_ok=True)
                if member.isdir():
                    target.mkdir(exist_ok=True)
                elif member.isfile():
                    with tar.extractfile(member) as source, open(target, 'wb') as output:
                        shutil.copyfileobj(source, output, 8 * 1024**2)
            for member in members:
                if member.issym():
                    (stage / member.name).symlink_to(member.linkname)
            for part in parts:
                if not (stage / part).is_dir():
                    raise ValueError('Prepared archive is incomplete')
            installed = []
            try:
                for part in parts:
                    target = runtime_path(BASE / part)
                    if target.exists():
                        target.rmdir()
                    (stage / part).rename(target)
                    installed.append(target)
            except Exception:
                for target in reversed(installed):
                    target.rename(stage / target.relative_to(BASE))
                raise
    return {mode: BASE / 'processed' / mode / 'data.yaml' for mode in settings['modes']}

for mode in MODES_TO_RUN:
    report = validate_labels(yaml_paths[mode].parent, mode)
    print({'mode': mode, 'data_yaml_valid': True, 'counts': report})
if not TERMS_REVIEWED:
    raise RuntimeError('利用条件を確認してください。')
if Path(DRIVE_MOUNT_POINT).absolute() not in Path(DRIVE_PREPARED_DIR).absolute().parents:
    raise ValueError('DRIVE_PREPARED_DIRはマウントしたGoogle Drive配下に指定してください。')
prepared_settings = {'modes': MODES_TO_RUN, 'debug': DEBUG_MODE, 'debug_limit': DEBUG_LIMIT if DEBUG_MODE else None}
print('手順12までの進捗をDriveへ保存中です。完了までお待ちください。')
prepared_archive = save_prepared_data(DRIVE_PREPARED_DIR, prepared_settings)
print('手順12までのDrive保存完了:', str(prepared_archive))

## 13. YOLOモデルの読み込み
参考実装のyolo26n.pt（設定でyolo26s.ptも選択可能）を使用します。
公式Ultralytics重みをランタイムに取得し、同じセッションではキャッシュを再利用します。
外部からアップロードした信頼できない.ptは読み込みません。
RESTORE_PREPARED_FROM_DRIVE=Trueの場合、Driveをマウントし、手順12の保存を照合・復元します。
新しいランタイムでは先に手順2〜5を実行してください。手順6〜12の再実行は不要です。
復元先は空のローカルデータディレクトリに限ります。保存時と異なるモード・DEBUG設定や破損した保存は拒否します。

In [ ]:
"""Public Hugging Face downloads, local validation and safe extraction."""
import json
import sys
import time
import traceback
import uuid
from datetime import datetime, timezone
import re
import shutil
import stat
import urllib.request
import urllib.error
from urllib.parse import urlsplit
import zipfile
import zlib
from pathlib import Path

HF_REPO_ID = 'sshao0516/CrowdHuman'
HF_REVISION = 'd97203da87e348ea69f7a7633a57c21a956120a6'
ARCHIVES = ('CrowdHuman_train01.zip', 'CrowdHuman_train02.zip', 'CrowdHuman_train03.zip', 'CrowdHuman_val.zip')
ANNOTATIONS = ('annotation_train.odgt', 'annotation_val.odgt')
EXPECTED = {'train': 15000, 'val': 4370}


class DownloadProgress:
    """Throttled text progress for a terminal or one updating notebook display."""
    def __init__(self, stream=None, interval=1.0, notebook=False):
        self.stream = sys.stderr if stream is None else stream
        self.interval = interval
        self.notebook = notebook
        self.last_update = None
        self.task = None
        self.started = None
        self.display = None

    def __call__(self, event):
        now = time.monotonic()
        stage = event['stage']
        task = (stage, event.get('filename'))
        changed = task != self.task or event.get('processed') == 0
        if changed:
            self.task, self.started = task, now
        finished = event.get('finished', False) or stage in ('reused', 'file_complete', 'complete', 'error', 'interrupted')
        if not changed and not finished and self.last_update is not None and now - self.last_update < self.interval:
            return
        self.last_update = now
        done = event.get('processed', 0)
        total = event.get('total', 0)
        prefix = f"[{event.get('completed_files', 0)}/{event.get('file_total', 6)} 完了]"
        filename = event.get('filename', '')
        labels = {'request': '接続中', 'validation': '検証中', 'transfer': 'ダウンロード中',
                  'copy': 'Driveからコピー中', 'checksum': 'チェックサム照合中',
                  'reused': '検証OK・再利用', 'file_complete': '取得・検証完了',
                  'complete': '全ファイル準備完了', 'error': '失敗', 'interrupted': '中断'}
        text = f"{prefix} {labels.get(stage, stage)} {filename}"
        if stage in ('transfer', 'validation', 'copy', 'checksum'):
            percent = f'{100 * done / total:5.1f}%' if total else '  -- %'
            elapsed = max(0.0, now - self.started)
            speed = done / elapsed if elapsed else 0
            remaining = max(0, total - done) / speed if total and speed else None
            eta = f'{int(remaining)//60:02d}:{int(remaining)%60:02d}' if remaining is not None else '--:--'
            if event.get('unit') == 'records':
                amount = f'{done:,}/{total:,} 件' if total else f'{done:,} 件'
                rate = f'{speed:,.0f} 件/s' if speed else '-- 件/s'
            else:
                divisor, unit = (1024**3, 'GiB') if max(done, total) >= 1024**3 else (1024**2, 'MiB')
                amount = f'{done/divisor:.1f}/{total/divisor:.1f} {unit}' if total else f'{done/divisor:.1f} {unit}'
                rate = f'{speed/1024**2:.1f} MiB/s' if speed else '-- MiB/s'
            text += f' | {percent} | {amount} | {rate} | 残り {eta}'
        if self.notebook:
            from html import escape
            from IPython.display import HTML, display
            content = HTML('<pre>' + escape(text) + '</pre>')
            if self.display is None:
                self.display = display(content, display_id=True)
            else:
                self.display.update(content)
        elif self.stream.isatty():
            print('\r' + text + '\033[K', end='\n' if finished else '', file=self.stream, flush=True)
        else:
            print(text, file=self.stream, flush=True)


def _notify_progress(progress, **event):
    if progress is not None:
        progress(event)


def _file_progress(progress, index, completed, total_files):
    if progress is None:
        return None
    return lambda event: progress({**event, 'file_index': index, 'completed_files': completed,
                                   'file_total': total_files})


def destination(name):
    if name not in ARCHIVES + ANNOTATIONS:
        raise ValueError('Unexpected dataset filename')
    return storage_root() / 'raw' / ('archives' if name in ARCHIVES else 'annotations') / name


def read_odgt(path):
    """Never include raw annotations or IDs in exception messages."""
    runtime_path(path)
    with open(path, encoding='utf-8') as handle:
        for line_no, line in enumerate(handle, 1):
            try:
                record = json.loads(line)
            except (ValueError, TypeError):
                raise ValueError(f'Invalid ODGT record at line {line_no}: invalid JSON') from None
            if not isinstance(record, dict):
                raise ValueError(f'Invalid ODGT record at line {line_no}: expected a JSON object')
            image_id = record.get('ID')
            # Original CrowdHuman image stems contain a comma. Keep separators,
            # dots, whitespace and control characters excluded from filesystem IDs.
            if not isinstance(image_id, str) or not re.fullmatch(r'[A-Za-z0-9_,\-]+', image_id):
                raise ValueError(f'Invalid ODGT record at line {line_no}: invalid image ID')
            if not isinstance(record.get('gtboxes'), list):
                raise ValueError(f'Invalid ODGT record at line {line_no}: gtboxes must be a list')
            yield record


def validate_file(path, name, progress=None):
    runtime_path(path)
    if not Path(path).is_file():
        raise FileNotFoundError(f'Missing required file: {name}')
    if name in ARCHIVES:
        try:
            with zipfile.ZipFile(path) as z:
                entries = z.infolist()
                if not entries:
                    raise ValueError('Empty ZIP')
                if progress is None:
                    if z.testzip() is not None:
                        raise ValueError('ZIP CRC check failed')
                else:
                    total = sum(entry.file_size for entry in entries if not entry.is_dir())
                    processed = 0
                    _notify_progress(progress, stage='validation', filename=name, processed=0, total=total, unit='bytes')
                    for entry in entries:
                        if entry.is_dir():
                            continue
                        with z.open(entry) as handle:
                            while chunk := handle.read(1024**2):
                                processed += len(chunk)
                                _notify_progress(progress, stage='validation', filename=name,
                                                 processed=processed, total=total, unit='bytes')
                    _notify_progress(progress, stage='validation', filename=name, processed=processed,
                                     total=total, unit='bytes', finished=True)
        except zipfile.BadZipFile:
            raise ValueError(f'Invalid ZIP: {name}') from None
        return None
    count = 0
    expected = EXPECTED['train' if name == 'annotation_train.odgt' else 'val']
    _notify_progress(progress, stage='validation', filename=name, processed=0, total=expected, unit='records')
    for _ in read_odgt(path):
        count += 1
        if count % 100 == 0:
            _notify_progress(progress, stage='validation', filename=name, processed=count, total=expected, unit='records')
    if count == 0:
        raise ValueError('Empty annotation file')
    _notify_progress(progress, stage='validation', filename=name, processed=count, total=count, unit='records', finished=True)
    return count


def huggingface_links(repo_id=HF_REPO_ID, revision=HF_REVISION):
    """Pin all six original files to one public dataset commit, without Hub upload APIs."""
    if not isinstance(repo_id, str) or not re.fullmatch(r'[A-Za-z0-9_-]+/[A-Za-z0-9_.-]+', repo_id):
        raise ValueError('Invalid Hugging Face dataset repository ID')
    if not isinstance(revision, str) or not re.fullmatch(r'[a-f0-9]{40}', revision):
        raise ValueError('Use a full Hugging Face dataset commit SHA as revision')
    root = f'https://huggingface.co/datasets/{repo_id}/resolve/{revision}'
    return {name: f'{root}/{name}?download=true' for name in ARCHIVES + ANNOTATIONS}


def _safe_download_message(message):
    """Never retain signed URLs, credentials, response bodies or annotation content."""
    text = re.sub(r'https?://[^\s\"\'<>]+', '[URL removed]', str(message))
    text = re.sub(r'(?i)\b(?:hf_[A-Za-z0-9]+|Bearer\s+[^\s,;]+)', '[credential removed]', text)
    text = re.sub(r'(?i)((?:token|password|authorization|signature|credential|api[_-]?key)\s*[:=]\s*)[^\s,;]+', r'\1[redacted]', text)
    return text[:1000]


def _download_event(log_path, context, event, **details):
    path = runtime_path(log_path)
    runtime_path(path.parent).mkdir(parents=True, exist_ok=True)
    entry = {'timestamp_utc': datetime.now(timezone.utc).isoformat(),
             'event': event, **context, **details}
    with open(path, 'a', encoding='utf-8') as handle:
        handle.write(json.dumps(entry, ensure_ascii=False) + '\n')


def _download_error_details(error):
    chain, seen = [], set()
    current = error
    while current is not None and id(current) not in seen:
        seen.add(id(current))
        item = {'type': type(current).__name__, 'message': _safe_download_message(current)}
        if isinstance(current, urllib.error.HTTPError):
            item['http_status'] = current.code
            item['reason'] = _safe_download_message(current.reason)
            if current.headers:
                item['response_headers'] = {
                    name: _safe_download_message(current.headers[name])
                    for name in ('X-Request-Id', 'Retry-After') if name in current.headers}
        elif isinstance(current, urllib.error.URLError):
            item['reason'] = _safe_download_message(current.reason)
        if isinstance(current, OSError) and current.errno is not None:
            item['errno'] = current.errno
        # Stack locations only: no source lines, locals, response payloads or data IDs.
        item['stack'] = [{'file': Path(frame.filename).name, 'line': frame.lineno,
                          'function': frame.name}
                         for frame in traceback.extract_tb(current.__traceback__)]
        chain.append(item)
        current = current.__cause__ if current.__cause__ is not None else current.__context__
    return chain


def download_huggingface(repo_id=HF_REPO_ID, revision=HF_REVISION, progress=None):
    """Download public files and retain sanitized diagnostics inside the runtime."""
    log_path = runtime_path(storage_root() / 'logs/download.jsonl')
    context = {'attempt_id': uuid.uuid4().hex, 'stage': 'configuration'}
    temp = None
    completed = 0
    total_files = len(ARCHIVES + ANNOTATIONS)
    try:
        links = huggingface_links(repo_id, revision)
        context.update(repo_id=repo_id, revision=revision)
        _download_event(log_path, context, 'start')
        needed = []
        for index, name in enumerate(ARCHIVES + ANNOTATIONS, 1):
            context.update(filename=name, stage='existing_file_validation')
            path = runtime_path(destination(name))
            if path.exists():
                validate_file(path, name, progress=_file_progress(progress, index, completed, total_files))
                completed += 1
                _notify_progress(progress, stage='reused', filename=name, completed_files=completed, file_total=total_files)
                _download_event(log_path, context, 'reused', bytes_on_disk=path.stat().st_size)
            else:
                needed.append(name)
        for name in needed:
            context = {'attempt_id': context['attempt_id'], 'repo_id': repo_id,
                       'revision': revision, 'filename': name, 'stage': 'http_request',
                       'bytes_received': 0}
            path = runtime_path(destination(name))
            temp = runtime_path(path.with_suffix(path.suffix + '.part'))
            index = (ARCHIVES + ANNOTATIONS).index(name) + 1
            current_progress = _file_progress(progress, index, completed, total_files)
            _notify_progress(current_progress, stage='request', filename=name)
            _download_event(log_path, context, 'file_start')
            # No auth headers, cookies, response bodies or signed redirect URLs are logged.
            with urllib.request.urlopen(links[name], timeout=120) as response, open(temp, 'wb') as out:
                context['http_status'] = getattr(response, 'status', None)
                context['response_host'] = urlsplit(response.geturl()).hostname if hasattr(response, 'geturl') else 'huggingface.co'
                content_type = response.headers.get('Content-Type', '')
                context['content_type'] = _safe_download_message(content_type)
                if 'text/html' in content_type.lower():
                    raise RuntimeError('Hugging Face returned an access page instead of a file')
                length = int(response.headers.get('Content-Length', 0))
                context.update(expected_bytes=length, stage='disk_space_check',
                               free_bytes=shutil.disk_usage(storage_root()).free)
                _download_event(log_path, context, 'response')
                if length and context['free_bytes'] < length + 2 * 1024**3:
                    raise OSError('Insufficient download disk space')
                context['stage'] = 'transfer'
                last_progress = 0
                _notify_progress(current_progress, stage='transfer', filename=name, processed=0, total=length, unit='bytes')
                while chunk := response.read(1024**2):
                    context['free_bytes'] = shutil.disk_usage(storage_root()).free
                    if context['free_bytes'] < len(chunk) + 1024**3:
                        raise OSError('Insufficient download disk space')
                    out.write(chunk)
                    context['bytes_received'] += len(chunk)
                    _notify_progress(current_progress, stage='transfer', filename=name,
                                     processed=context['bytes_received'], total=length, unit='bytes')
                    if context['bytes_received'] - last_progress >= 64 * 1024**2:
                        _download_event(log_path, context, 'progress')
                        last_progress = context['bytes_received']
                if length and context['bytes_received'] != length:
                    raise ValueError('Incomplete Hugging Face response')
            _notify_progress(current_progress, stage='transfer', filename=name,
                             processed=context['bytes_received'], total=length, unit='bytes', finished=True)
            context['stage'] = 'downloaded_file_validation'
            validate_file(temp, name, progress=current_progress)
            temp.replace(path)
            temp = None
            completed += 1
            _notify_progress(progress, stage='file_complete', filename=name, completed_files=completed, file_total=total_files)
            _download_event(log_path, context, 'file_complete')
        context['stage'] = 'save_source_metadata'
        source = runtime_path(storage_root() / 'raw/download_source.json')
        source.write_text(json.dumps({'requested_repo_id': repo_id, 'requested_revision': revision,
                                      'downloaded': needed,
                                      'reused_existing': [name for name in ARCHIVES + ANNOTATIONS if name not in needed]}, indent=2))
        _download_event(log_path, context, 'complete', downloaded_files=len(needed),
                        reused_files=len(ARCHIVES + ANNOTATIONS) - len(needed))
        _notify_progress(progress, stage='complete', completed_files=completed, file_total=total_files)
        return log_path
    except (Exception, KeyboardInterrupt) as error:
        cleanup_error = None
        if temp is not None:
            try:
                temp.unlink(missing_ok=True)
            except OSError as cleanup:
                cleanup_error = _safe_download_message(cleanup)
        log_note = f'Log: {log_path}'
        try:
            _download_event(log_path, context, 'interrupted' if isinstance(error, KeyboardInterrupt) else 'error',
                            errors=_download_error_details(error), cleanup_error=cleanup_error)
        except OSError as log_error:
            log_note = f'Could not write log {log_path}: {_safe_download_message(log_error)}'
        _notify_progress(progress, stage='interrupted' if isinstance(error, KeyboardInterrupt) else 'error',
                         filename=context.get('filename', ''), completed_files=completed, file_total=total_files)
        if isinstance(error, KeyboardInterrupt):
            raise
        raise RuntimeError(f'Hugging Face download failed: {context.get("filename", "configuration")}; '
                           f'{type(error).__name__}: {_safe_download_message(error)}. '
                           + log_note) from None


def upload_missing():
    """Colab upload widget writes files directly to raw; it never prints file contents."""
    from google.colab import files
    import os
    previous = Path.cwd()
    try:
        os.chdir(runtime_path(storage_root() / 'raw'))
        uploaded = files.upload()
        del uploaded  # Browser upload may use substantial RAM for large archives.
        for name in ARCHIVES + ANNOTATIONS:
            staged = storage_root() / 'raw' / name
            if staged.exists():
                target = destination(name)
                if target.exists():
                    validate_file(target, name)
                    staged.unlink()
                else:
                    validate_file(staged, name)
                    staged.replace(target)
    finally:
        os.chdir(previous)


def inventory(progress=None):
    counts = {}
    files = ARCHIVES + ANNOTATIONS
    for index, name in enumerate(files, 1):
        count = validate_file(destination(name), name, progress=_file_progress(progress, index, index - 1, len(files)))
        if count is not None:
            counts[name] = count
    for split in EXPECTED:
        if counts[f'annotation_{split}.odgt'] != EXPECTED[split]:
            raise ValueError('Official annotation count mismatch')
    _notify_progress(progress, stage='complete', completed_files=len(files), file_total=len(files))
    return {'annotation_counts': counts, 'free_gib': round(shutil.disk_usage(storage_root()).free / 1024**3, 2)}


def extract_archives():
    root = runtime_path(storage_root() / 'raw/images')
    entries = []
    seen = set()
    for name in ARCHIVES:
        archive = destination(name)
        validate_file(archive, name)
        with zipfile.ZipFile(archive) as z:
            for item in z.infolist():
                if item.is_dir():
                    continue
                parts = Path(item.filename).parts
                if len(parts) != 2 or parts[0] != 'Images' or Path(parts[1]).suffix.lower() not in ('.jpg', '.jpeg'):
                    raise ValueError('Unexpected archive member')
                if stat.S_ISLNK(item.external_attr >> 16) or parts[1] in seen:
                    raise ValueError('Duplicate/unsafe archive member')
                seen.add(parts[1])
                target = runtime_path(root / parts[1])
                entries.append((archive, item.filename, target, item.file_size, item.CRC))
    required = sum(size for _, _, p, size, _ in entries if not p.exists() or p.stat().st_size != size)
    if shutil.disk_usage(storage_root()).free < required + 2 * 1024**3:
        raise OSError('Insufficient extraction space (plus 2 GiB reserve)')
    # Compare CRCs before reuse; a same-size corrupt image must also be replaced.
    for archive in map(destination, ARCHIVES):
        with zipfile.ZipFile(archive) as z:
            for source_archive, member, target, size, crc in entries:
                if source_archive != archive:
                    continue
                if target.exists() and target.stat().st_size == size:
                    actual = 0
                    with open(target, 'rb') as existing:
                        for chunk in iter(lambda: existing.read(1024**2), b''):
                            actual = zlib.crc32(chunk, actual)
                    if actual == crc:
                        continue
                part = target.with_suffix('.part')
                with z.open(member) as source, open(part, 'wb') as out:
                    shutil.copyfileobj(source, out)
                part.replace(target)
    return {'archive_images': len(entries)}


def image_size(path):
    runtime_path(path)
    from PIL import Image
    try:
        with Image.open(path) as image:
            size = image.size
            image.verify()
        if min(size) <= 0:
            raise ValueError()
        return size
    except Exception:
        raise ValueError('Missing or corrupt dataset image') from None


def inspect_dataset():
    """Check every image, including DEBUG runs; reject split overlap and duplicates."""
    splits = {}
    sizes = {}
    for split in ('train', 'val'):
        ids = set()
        for record in read_odgt(destination(f'annotation_{split}.odgt')):
            key = record['ID']
            if key in ids:
                raise ValueError('Duplicate image ID within split')
            ids.add(key)
            sizes[key] = image_size(storage_root() / 'raw/images' / f'{key}.jpg')
        splits[split] = ids
    if splits['train'] & splits['val']:
        raise ValueError('Train/val overlap detected')
    return sizes, {split: len(ids) for split, ids in splits.items()}


def _copy_checksum(path, name, progress=None):
    """Read only an explicitly approved source/cache file; never expose its contents."""
    import hashlib
    digest = hashlib.sha256()
    total = path.stat().st_size
    processed = 0
    _notify_progress(progress, stage='checksum', filename=name, processed=0, total=total, unit='bytes')
    with open(path, 'rb') as handle:
        while chunk := handle.read(1024**2):
            digest.update(chunk)
            processed += len(chunk)
            _notify_progress(progress, stage='checksum', filename=name, processed=processed, total=total, unit='bytes')
    _notify_progress(progress, stage='checksum', filename=name, processed=processed, total=total, unit='bytes', finished=True)
    return digest.hexdigest()


def copy_drive_dataset(source_dir, progress=None):
    """Copy six private Drive cache files into the active runtime before processing.

    Drive layout matches raw/archives and raw/annotations. Sources are read only;
    all labels, images, training runs and temporary copy files stay in the runtime.
    """
    import hashlib
    source_root = Path(source_dir).expanduser().absolute()
    if source_root.resolve() != source_root:
        raise ValueError('Drive dataset directory must not traverse symlinks')
    if source_root == storage_root():
        raise ValueError('Drive source and runtime destination must differ')
    files = ARCHIVES + ANNOTATIONS
    log_path = runtime_path(storage_root() / 'logs/drive_copy.jsonl')
    context = {'attempt_id': uuid.uuid4().hex, 'stage': 'source_inventory'}
    temporary = None
    completed = 0
    try:
        sources = {}
        for name in files:
            relative = Path('raw') / ('archives' if name in ARCHIVES else 'annotations') / name
            source = source_root / relative
            if source_root not in source.resolve().parents:
                raise ValueError('Drive source file escapes its dataset directory')
            if not source.is_file():
                raise FileNotFoundError(f'Missing Drive cache file: {name}')
            sources[name] = source
        required = sum(source.stat().st_size for name, source in sources.items() if not destination(name).exists())
        if shutil.disk_usage(storage_root()).free < required + 2 * 1024**3:
            raise OSError('Insufficient runtime space for Drive copy (plus 2 GiB reserve)')
        _download_event(log_path, context, 'start', file_total=len(files), required_bytes=required)
        records = []
        for index, name in enumerate(files, 1):
            source = sources[name]
            target = runtime_path(destination(name))
            runtime_path(target.parent).mkdir(parents=True, exist_ok=True)
            current_progress = _file_progress(progress, index, completed, len(files))
            context.update(stage='copy', filename=name, bytes_received=0, expected_bytes=source.stat().st_size)
            if target.exists():
                context['stage'] = 'existing_file_checksum'
                source_hash = _copy_checksum(source, name, current_progress)
                if _copy_checksum(target, name, current_progress) != source_hash:
                    raise ValueError(f'Runtime file differs from Drive cache: {name}; remove the local file explicitly before retrying')
                validate_file(target, name, progress=current_progress)
                action = 'reused'
            else:
                temporary = runtime_path(target.with_suffix(target.suffix + '.copying'))
                digest = hashlib.sha256()
                _notify_progress(current_progress, stage='copy', filename=name, processed=0,
                                 total=context['expected_bytes'], unit='bytes')
                with open(source, 'rb') as incoming, open(temporary, 'wb') as outgoing:
                    while chunk := incoming.read(1024**2):
                        if shutil.disk_usage(storage_root()).free < len(chunk) + 1024**3:
                            raise OSError('Insufficient runtime space during Drive copy')
                        outgoing.write(chunk)
                        digest.update(chunk)
                        context['bytes_received'] += len(chunk)
                        _notify_progress(current_progress, stage='copy', filename=name,
                                         processed=context['bytes_received'], total=context['expected_bytes'], unit='bytes')
                if context['bytes_received'] != context['expected_bytes']:
                    raise ValueError('Drive source changed or copy was incomplete')
                _notify_progress(current_progress, stage='copy', filename=name,
                                 processed=context['bytes_received'], total=context['expected_bytes'], unit='bytes', finished=True)
                source_hash = digest.hexdigest()
                context['stage'] = 'copied_file_checksum'
                if _copy_checksum(temporary, name, current_progress) != source_hash:
                    raise ValueError('Drive copy checksum mismatch')
                context['stage'] = 'copied_file_validation'
                validate_file(temporary, name, progress=current_progress)
                temporary.replace(target)
                temporary = None
                action = 'copied'
            completed += 1
            records.append({'filename': name, 'sha256': source_hash, 'action': action})
            _download_event(log_path, context, action, sha256=source_hash, completed_files=completed)
            _notify_progress(progress, stage='reused' if action == 'reused' else 'file_complete',
                             filename=name, completed_files=completed, file_total=len(files))
        manifest = runtime_path(storage_root() / 'raw/drive_copy_source.json')
        manifest.write_text(json.dumps({'source_directory': str(source_root), 'files': records}, indent=2))
        context['stage'] = 'complete'
        _download_event(log_path, context, 'complete', completed_files=completed)
        _notify_progress(progress, stage='complete', completed_files=completed, file_total=len(files))
        return log_path
    except (Exception, KeyboardInterrupt) as error:
        if temporary is not None:
            temporary.unlink(missing_ok=True)
        _download_event(log_path, context, 'interrupted' if isinstance(error, KeyboardInterrupt) else 'error',
                        errors=_download_error_details(error))
        _notify_progress(progress, stage='interrupted' if isinstance(error, KeyboardInterrupt) else 'error',
                         filename=context.get('filename', ''), completed_files=completed, file_total=len(files))
        if isinstance(error, KeyboardInterrupt):
            raise
        raise RuntimeError(f'Drive copy failed: {_safe_download_message(error)}. Log: {log_path}') from None

"""Separate body/head ignore policy and streaming YOLO label conversion."""
import hashlib
import json
import math
import shutil
from collections import Counter
from pathlib import Path

NAMES = {'vbox': {0: 'person'}, 'hbox': {0: 'head'}, 'both': {0: 'person', 1: 'head'}}


def normalize_box(box, width, height):
    if not isinstance(box, (list, tuple)) or len(box) != 4:
        raise ValueError('Invalid bbox')
    if any(isinstance(v, bool) or not isinstance(v, (int, float)) or not math.isfinite(v) for v in box):
        raise ValueError('Invalid bbox')
    x, y, w, h = box
    if width <= 0 or height <= 0 or w <= 0 or h <= 0:
        raise ValueError('Nonpositive dimensions')
    x1, y1 = max(0, min(width, x)), max(0, min(height, y))
    x2, y2 = max(0, min(width, x + w)), max(0, min(height, y + h))
    if x2 <= x1 or y2 <= y1:
        raise ValueError('Empty clipped bbox')
    return ((x1 + x2) / (2 * width), (y1 + y2) / (2 * height), (x2 - x1) / width, (y2 - y1) / height)


def ignored(annotation, attr):
    value = annotation.get(attr, {})
    if not isinstance(value, dict) or value.get('ignore', 0) not in (0, 1):
        raise ValueError('Invalid ignore attribute')
    return value.get('ignore', 0) == 1


def convert_boxes(boxes, mode, width, height):
    if mode not in NAMES:
        raise ValueError('Unsupported mode')
    labels, stats = [], Counter()
    fields = ('vbox', 'hbox') if mode == 'both' else (mode,)
    for annotation in boxes:
        if not isinstance(annotation, dict):
            stats['invalid_annotation'] += 1
            continue
        if annotation.get('tag') != 'person':
            stats['non_person'] += 1
            continue
        for field in fields:
            try:
                if ignored(annotation, 'extra' if field == 'vbox' else 'head_attr'):
                    stats[f'{field}_ignored'] += 1
                    continue
                box = annotation.get(field)
                normalized = normalize_box(box, width, height)
                class_id = 1 if mode == 'both' and field == 'hbox' else 0
                labels.append((class_id, *normalized))
                stats[f'{field}_accepted'] += 1
                x, y, w, h = box
                if x < 0 or y < 0 or x + w > width or y + h > height:
                    stats[f'{field}_clipped'] += 1
            except (ValueError, TypeError):
                stats[f'{field}_invalid'] += 1
    return labels, stats


def fingerprint(sizes, limit):
    digest = hashlib.sha256()
    for split in ('train', 'val'):
        with open(destination(f'annotation_{split}.odgt'), 'rb') as handle:
            for chunk in iter(lambda: handle.read(1024**2), b''):
                digest.update(chunk)
    digest.update(json.dumps(sorted(sizes.items())).encode())
    digest.update(str(limit).encode())
    return digest.hexdigest()


def convert_dataset(mode, sizes, limit=None):
    if mode not in NAMES or (limit is not None and limit <= 0):
        raise ValueError('Invalid conversion configuration')
    root = runtime_path(BASE / 'processed' / mode)
    signature = fingerprint(sizes, limit)
    manifest = root / 'conversion.json'
    if manifest.exists():
        previous = json.loads(manifest.read_text())
        if previous['fingerprint'] == signature:
            validate_labels(root, mode)
            return root / 'data.yaml'
        raise ValueError('Dataset changed; remove processed mode and matching run explicitly')
    staging = root.with_name(mode + '.building')
    if staging.exists():
        shutil.rmtree(runtime_path(staging))
    counts = Counter()
    selected = set()
    try:
        for split in ('train', 'val'):
            images = staging / 'images' / split
            labels_dir = staging / 'labels' / split
            images.mkdir(parents=True)
            labels_dir.mkdir(parents=True)
            for index, record in enumerate(read_odgt(destination(f'annotation_{split}.odgt'))):
                if limit is not None and index >= limit:
                    break
                key = record['ID']
                if key in selected:
                    raise ValueError('Duplicate image or split overlap')
                selected.add(key)
                width, height = sizes[key]
                source = runtime_path(BASE / 'raw/images' / f'{key}.jpg')
                if not source.is_file():
                    raise FileNotFoundError('Missing source image')
                rows, stats = convert_boxes(record['gtboxes'], mode, width, height)
                counts.update({f'{split}_{k}': v for k, v in stats.items()})
                counts[f'{split}_images'] += 1
                (images / source.name).symlink_to(source)
                (labels_dir / f'{key}.txt').write_text(''.join(' '.join([str(row[0]), *(f'{v:.10f}' for v in row[1:])]) + '\n' for row in rows))
        if not counts['train_images'] or not counts['val_images']:
            raise ValueError('Both splits must be nonempty')
        (staging / 'conversion.json').write_text(json.dumps({'fingerprint': signature, 'limit': limit, 'counts': counts}, indent=2))
        if root.exists():
            raise ValueError('Incomplete processed directory; remove it explicitly')
        staging.rename(root)
        write_yaml(root, mode)
        validate_labels(root, mode)
    except Exception:
        if staging.exists():
            shutil.rmtree(staging)
        raise
    return root / 'data.yaml'


def write_yaml(root, mode):
    import yaml
    root = runtime_path(root)
    data = {'path': str(root), 'train': 'images/train', 'val': 'images/val', 'names': NAMES[mode]}
    (root / 'data.yaml').write_text(yaml.safe_dump(data, sort_keys=False))


def validate_labels(root, mode):
    import yaml
    root = runtime_path(root)
    data = yaml.safe_load((root / 'data.yaml').read_text())
    if data != {'path': str(root), 'train': 'images/train', 'val': 'images/val', 'names': NAMES[mode]}:
        raise ValueError('data.yaml mismatch')
    ids = []
    totals = {}
    for split in ('train', 'val'):
        images = list((root / 'images' / split).glob('*.jpg'))
        label_files = list((root / 'labels' / split).glob('*.txt'))
        if not images or {p.stem for p in images} != {p.stem for p in label_files}:
            raise ValueError('Images/labels mismatch')
        boxes = 0
        ids.append({p.stem for p in images})
        for image in images:
            runtime_path(image)
            if not image.is_file():
                raise FileNotFoundError('Broken image symlink')
        for path in label_files:
            runtime_path(path)
            for line in path.read_text().splitlines():
                fields = line.split()
                if len(fields) != 5 or fields[0] not in map(str, NAMES[mode]):
                    raise ValueError('Invalid class or label columns')
                values = list(map(float, fields[1:]))
                x, y, w, h = values
                if not all(math.isfinite(v) and 0 <= v <= 1 for v in values) or min(w, h) <= 0:
                    raise ValueError('Invalid normalized label')
                if x-w/2 < -1e-8 or x+w/2 > 1+1e-8 or y-h/2 < -1e-8 or y+h/2 > 1+1e-8:
                    raise ValueError('Label crosses image boundary')
                boxes += 1
        if not boxes:
            raise ValueError('Split has no usable labels')
        totals[split] = {'images': len(images), 'boxes': boxes}
    if ids[0] & ids[1]:
        raise ValueError('Split overlap')
    return totals

"""Save prepared private data once; restore without downloading or converting."""
import hashlib
import json
import os
import shutil
import tarfile
import tempfile
import uuid
from pathlib import Path, PurePosixPath


def _prepared_hash(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(8 * 1024**2), b''):
            digest.update(chunk)
    return digest.hexdigest()


def _prepared_root(directory):
    root = Path(directory).absolute()
    if root.resolve() != root:
        raise ValueError('Prepared data directory must not traverse symlinks')
    return root


def _prepared_parts(settings):
    modes = settings['modes']
    if not modes or len(set(modes)) != len(modes) or any(m not in ('vbox', 'hbox', 'both') for m in modes):
        raise ValueError('Invalid prepared modes')
    return ['raw/images', 'raw/annotations'] + ['processed/' + mode for mode in modes]


def save_prepared_data(directory, settings):
    """Publish latest.json only after a complete archive has been verified."""
    root = _prepared_root(directory)
    parts = _prepared_parts(settings)
    root.mkdir(parents=True, exist_ok=True)
    name = 'prepared_' + uuid.uuid4().hex + '.tar'
    archive = root / name
    temporary = root / (name + '.incomplete')
    try:
        with tarfile.open(temporary, 'w') as tar:
            for part in parts:
                source = runtime_path(BASE / part)
                if not source.is_dir():
                    raise FileNotFoundError('Prepared data is missing: ' + part)
                for path in [source, *sorted(source.rglob('*'))]:
                    relative = path.relative_to(BASE).as_posix()
                    runtime_path(path)
                    info = tar.gettarinfo(str(path), arcname=relative)
                    if info.issym():
                        target = path.resolve()
                        image_root = BASE / 'raw/images'
                        if image_root not in target.parents or not target.is_file():
                            raise ValueError('Prepared link must point to an image')
                        info.linkname = os.path.relpath(target, path.parent)
                        tar.addfile(info)
                    elif info.isfile():
                        with open(path, 'rb') as handle:
                            tar.addfile(info, handle)
                    elif info.isdir():
                        tar.addfile(info)
                    else:
                        raise ValueError('Unsupported prepared file')
        checksum = _prepared_hash(temporary)
        temporary.rename(archive)
        manifest = {'schema': 1, 'archive': name, 'sha256': checksum,
                    'size': archive.stat().st_size, 'settings': settings}
        pointer = root / ('latest.' + uuid.uuid4().hex + '.tmp')
        pointer.write_text(json.dumps(manifest, ensure_ascii=False, indent=2))
        pointer.replace(root / 'latest.json')
        return archive
    finally:
        temporary.unlink(missing_ok=True)


def restore_prepared_data(directory, settings):
    """Verify paths and checksum, extract in staging, then install into empty dirs."""
    root = _prepared_root(directory)
    parts = _prepared_parts(settings)
    manifest = json.loads((root / 'latest.json').read_text())
    if manifest.get('schema') != 1 or manifest.get('settings') != settings:
        raise ValueError('Prepared data settings mismatch; use the saved modes/DEBUG settings')
    name = manifest['archive']
    if not isinstance(name, str) or not name.startswith('prepared_') or not name.endswith('.tar') or Path(name).name != name:
        raise ValueError('Invalid prepared archive name')
    archive = _prepared_root(root / name)
    if archive.stat().st_size != manifest['size'] or _prepared_hash(archive) != manifest['sha256']:
        raise ValueError('Prepared archive checksum mismatch')
    for part in parts:
        target = runtime_path(BASE / part)
        if target.exists() and (not target.is_dir() or any(target.iterdir())):
            raise FileExistsError('Restore requires an empty local directory: ' + part)
    with tarfile.open(archive, 'r:') as tar:
        members = tar.getmembers()
        seen, links = set(), set()
        for member in members:
            path = PurePosixPath(member.name)
            if path.is_absolute() or '..' in path.parts or member.name in seen:
                raise ValueError('Unsafe prepared archive path')
            if not any(member.name == part or member.name.startswith(part + '/') for part in parts):
                raise ValueError('Unexpected prepared archive path')
            seen.add(member.name)
            if member.issym():
                target = (BASE / member.name).parent / member.linkname
                resolved = target.resolve()
                if (BASE / 'raw/images') not in resolved.parents or Path(member.linkname).is_absolute():
                    raise ValueError('Unsafe prepared archive link')
                links.add(member.name)
            elif not (member.isfile() or member.isdir()):
                raise ValueError('Unsupported prepared archive entry')
        for member in members:
            if any(parent.as_posix() in links for parent in PurePosixPath(member.name).parents):
                raise ValueError('Prepared archive traverses a link')
        required = sum(member.size for member in members if member.isfile())
        if shutil.disk_usage(BASE).free < required + 1024**2:
            raise OSError('Insufficient local space for prepared data')
        with tempfile.TemporaryDirectory(prefix='.prepared-', dir=BASE) as temporary:
            stage = Path(temporary)
            # Extract regular entries first; links are created only after all writes.
            for member in members:
                target = stage / member.name
                target.parent.mkdir(parents=True, exist_ok=True)
                if member.isdir():
                    target.mkdir(exist_ok=True)
                elif member.isfile():
                    with tar.extractfile(member) as source, open(target, 'wb') as output:
                        shutil.copyfileobj(source, output, 8 * 1024**2)
            for member in members:
                if member.issym():
                    (stage / member.name).symlink_to(member.linkname)
            for part in parts:
                if not (stage / part).is_dir():
                    raise ValueError('Prepared archive is incomplete')
            installed = []
            try:
                for part in parts:
                    target = runtime_path(BASE / part)
                    if target.exists():
                        target.rmdir()
                    (stage / part).rename(target)
                    installed.append(target)
            except Exception:
                for target in reversed(installed):
                    target.rename(stage / target.relative_to(BASE))
                raise
    return {mode: BASE / 'processed' / mode / 'data.yaml' for mode in settings['modes']}

if RESTORE_PREPARED_FROM_DRIVE:
    if not TERMS_REVIEWED:
        raise RuntimeError('利用条件を確認し、TERMS_REVIEWED=Trueにしてください。')
    from google.colab import drive
    drive.mount(DRIVE_MOUNT_POINT)
    if Path(DRIVE_MOUNT_POINT).absolute() not in Path(DRIVE_PREPARED_DIR).absolute().parents:
        raise ValueError('DRIVE_PREPARED_DIRはマウントしたGoogle Drive配下に指定してください。')
    prepared_settings = {'modes': MODES_TO_RUN, 'debug': DEBUG_MODE, 'debug_limit': DEBUG_LIMIT if DEBUG_MODE else None}
    print('手順12の保存を照合・復元中です。完了までお待ちください。')
    yaml_paths = restore_prepared_data(DRIVE_PREPARED_DIR, prepared_settings)
    for mode in MODES_TO_RUN:
        validate_labels(yaml_paths[mode].parent, mode)
    print('手順12の保存を復元しました。13から続行します。')

"""Verified private Drive snapshots, committed only after every file has copied."""
import hashlib
import json
import re
import shutil
import uuid
from pathlib import Path

CHECKPOINT_FILES = ('weights/last.pt', 'weights/best.pt', 'experiment.json', 'results.csv', 'args.yaml')


def _checkpoint_hash(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(1024**2), b''):
            digest.update(chunk)
    return digest.hexdigest()


def _checkpoint_drive_path(path, root):
    path = Path(path).absolute()
    if root.resolve() != root or root not in path.resolve().parents:
        raise ValueError('Checkpoint path escapes its private Drive directory')
    return path


def drive_experiment_dir(config, drive_runs_dir):
    config.validate()
    root = Path(drive_runs_dir).expanduser().absolute()
    if root.resolve() != root:
        raise ValueError('Drive checkpoint root must not traverse symlinks')
    return _checkpoint_drive_path(root / config.mode / config.run_dir.name, root)


def save_drive_checkpoint(config, drive_runs_dir, epoch):
    """Preserve an unstripped optimizer checkpoint before Ultralytics final_eval."""
    if not isinstance(epoch, int) or epoch < 1:
        raise ValueError('Checkpoint epoch must be positive')
    run = config.run_dir
    experiment = json.loads(runtime_path(run / 'experiment.json').read_text())
    if not runtime_path(run / 'weights/last.pt').is_file():
        raise FileNotFoundError('No last.pt to back up')
    root = drive_experiment_dir(config, drive_runs_dir)
    root.mkdir(parents=True, exist_ok=True)
    snapshot_name = f'epoch_{epoch:06d}_{uuid.uuid4().hex[:8]}'
    staging = _checkpoint_drive_path(root / (snapshot_name + '.incomplete'), root)
    final = _checkpoint_drive_path(root / snapshot_name, root)
    staging.mkdir()
    try:
        files = {}
        for relative in CHECKPOINT_FILES:
            source = runtime_path(run / relative)
            if not source.exists():
                continue
            target = _checkpoint_drive_path(staging / relative, root)
            target.parent.mkdir(parents=True, exist_ok=True)
            before = _checkpoint_hash(source)
            shutil.copyfile(source, target)
            if _checkpoint_hash(target) != before:
                raise ValueError('Drive checkpoint copy checksum mismatch')
            files[relative] = {'sha256': before, 'size': target.stat().st_size}
        manifest = {'epoch': epoch, 'specification': experiment['specification'], 'files': files}
        manifest_path = staging / 'manifest.json'
        manifest_path.write_text(json.dumps(manifest, indent=2))
        manifest_hash = _checkpoint_hash(manifest_path)
        staging.replace(final)
        # The previous latest pointer remains valid until the entire snapshot is ready.
        pointer = _checkpoint_drive_path(root / 'latest.json', root)
        temporary_pointer = _checkpoint_drive_path(root / ('latest.' + uuid.uuid4().hex + '.tmp'), root)
        try:
            temporary_pointer.write_text(json.dumps({'snapshot': snapshot_name, 'epoch': epoch,
                                                     'manifest_sha256': manifest_hash}, indent=2))
            temporary_pointer.replace(pointer)
        finally:
            temporary_pointer.unlink(missing_ok=True)
    except BaseException:
        if staging.exists():
            shutil.rmtree(staging)
        raise
    return {'epoch': epoch, 'directory': str(final)}


def restore_drive_checkpoint(config, drive_runs_dir, specification):
    """Restore latest complete, matching snapshot into a fresh Colab run directory."""
    root = drive_experiment_dir(config, drive_runs_dir)
    pointer = json.loads(_checkpoint_drive_path(root / 'latest.json', root).read_text())
    name = pointer.get('snapshot', '')
    if not isinstance(name, str) or not re.fullmatch(r'epoch_[0-9]{6,}_[a-f0-9]{8}', name):
        raise ValueError('Invalid Drive snapshot pointer')
    snapshot = _checkpoint_drive_path(root / name, root)
    manifest_path = _checkpoint_drive_path(snapshot / 'manifest.json', root)
    if _checkpoint_hash(manifest_path) != pointer.get('manifest_sha256'):
        raise ValueError('Drive checkpoint manifest checksum mismatch')
    manifest = json.loads(manifest_path.read_text())
    if manifest.get('specification') != specification:
        raise ValueError('Drive checkpoint configuration/dataset/library mismatch')
    if manifest.get('epoch') != pointer.get('epoch'):
        raise ValueError('Drive checkpoint epoch mismatch')
    files = manifest.get('files', {})
    if not isinstance(files, dict) or not {'weights/last.pt', 'experiment.json'} <= files.keys() or not files.keys() <= set(CHECKPOINT_FILES):
        raise ValueError('Invalid Drive checkpoint file list')
    run = config.run_dir
    if run.exists() and any(run.iterdir()):
        raise FileExistsError('Restore requires a missing or empty local run directory')
    staging = runtime_path(run.with_name(run.name + '.restore-' + uuid.uuid4().hex))
    staging.mkdir(parents=True)
    try:
        for relative, expected in files.items():
            source = _checkpoint_drive_path(snapshot / relative, root)
            target = runtime_path(staging / relative)
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(source, target)
            if target.stat().st_size != expected['size'] or _checkpoint_hash(target) != expected['sha256']:
                raise ValueError('Drive checkpoint file checksum mismatch')
        meta = json.loads((staging / 'experiment.json').read_text())
        if meta.get('specification') != specification:
            raise ValueError('Drive experiment metadata mismatch')
        for session in meta.get('sessions', []):
            if session.get('status') == 'running':
                session['status'] = 'interrupted'
        (staging / 'experiment.json').write_text(json.dumps(meta, indent=2))
        if run.exists():
            run.rmdir()  # Only an empty directory was allowed above.
        staging.replace(run)
    except BaseException:
        if staging.exists():
            shutil.rmtree(staging)
        raise
    return runtime_path(run / 'weights/last.pt')

"""Local-only, reproducible training with explicit checkpoint resumption."""
import importlib.metadata
import json
import os
import random
import time
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path


@dataclass(frozen=True)
class TrainConfig:
    mode: str = 'vbox'
    model: str = 'yolo26n.pt'
    epochs: int = 50
    batch: int = 16
    imgsz: int = 640
    seed: int = 42
    patience: int = 10
    workers: int = 2
    debug: bool = False
    run_name: str = 'experiment'
    allow_cpu: bool = False

    def validate(self):
        if self.mode not in NAMES or min(self.epochs, self.batch, self.imgsz) <= 0:
            raise ValueError('Invalid training configuration')
        if self.imgsz % 32 or self.workers < 0 or self.patience < 0:
            raise ValueError('Invalid image size, workers or patience')
        if self.model not in ('yolo26n.pt', 'yolo26s.pt'):
            raise ValueError('Use the reference YOLO26n/s model')
        if not self.run_name or any(c not in 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-' for c in self.run_name):
            raise ValueError('Unsafe run name')

    @property
    def run_dir(self):
        self.validate()
        return runtime_path(BASE / 'runs' / self.mode / (self.run_name + ('_debug' if self.debug else '')))


def versions():
    names = ('ultralytics', 'torch', 'torchvision', 'Pillow', 'PyYAML', 'onnx', 'onnxruntime')
    result = {}
    for name in names:
        try:
            result[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            result[name] = 'not installed'
    return result


def device_for(config):
    import torch
    if torch.cuda.is_available():
        return 0
    warnings.warn('GPUがありません。ColabのGPUランタイムを選択してください。', stacklevel=2)
    if not config.allow_cpu:
        raise RuntimeError('GPU required; set allow_cpu=True only for a deliberate CPU test')
    return 'cpu'


def load_model(config):
    config.validate()
    YOLO = private_yolo()
    previous = Path.cwd()
    try:
        os.chdir(runtime_path(BASE / 'weights'))
        return YOLO(str(BASE / 'weights' / config.model))
    finally:
        os.chdir(previous)


def train(config, resume=False, drive_runs_dir=None, checkpoint_status=None):
    config.validate()
    import numpy as np
    import torch
    device = device_for(config)
    random.seed(config.seed)
    np.random.seed(config.seed)
    torch.manual_seed(config.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(config.seed)
    root = runtime_path(BASE / 'processed' / config.mode)
    validate_labels(root, config.mode)
    conversion = json.loads((root / 'conversion.json').read_text())
    if (conversion['limit'] is not None) != config.debug:
        raise ValueError('DEBUG flag and converted data disagree')
    run = config.run_dir
    meta_path = run / 'experiment.json'
    specification = {'config': asdict(config), 'dataset_fingerprint': conversion['fingerprint'], 'versions': versions()}
    if drive_runs_dir is not None:
        drive_run = drive_experiment_dir(config, drive_runs_dir)
        if not resume and (drive_run / 'latest.json').exists():
            raise FileExistsError('Drive experiment exists; resume it or select a new run_name')
        if resume and not meta_path.exists():
            restore_drive_checkpoint(config, drive_runs_dir, specification)
    if meta_path.exists():
        meta = json.loads(meta_path.read_text())
        if not resume:
            raise FileExistsError('Run exists; explicitly resume or select a new run_name')
        if meta['specification'] != specification:
            raise ValueError('Checkpoint configuration/dataset/library mismatch')
    else:
        if resume or run.exists():
            raise ValueError('No matching resumable experiment metadata')
        run.mkdir(parents=True)
        meta = {'specification': specification, 'sessions': []}
        meta_path.write_text(json.dumps(meta, indent=2))
    checkpoint = runtime_path(run / 'weights/last.pt')
    YOLO = private_yolo()
    if resume:
        if not checkpoint.is_file():
            raise FileNotFoundError('Missing last.pt checkpoint')
        # Only our trusted runtime checkpoint; never load arbitrary uploaded pickle files.
        ckpt = torch.load(checkpoint, map_location='cpu', weights_only=False)
        if ckpt.get('epoch', -1) < 0 or ckpt.get('optimizer') is None:
            raise ValueError('Completed/stripped checkpoint cannot resume; restore the raw Drive snapshot into a fresh local run')
        if ckpt['epoch'] + 1 >= config.epochs:
            raise ValueError('Checkpoint already reached the configured final epoch')
        del ckpt
    started = time.perf_counter()
    status = 'interrupted'
    session = {'seconds': 0.0, 'status': 'running', 'resume': resume}
    meta['sessions'].append(session)
    meta_path.write_text(json.dumps(meta, indent=2))

    def persist_epoch(trainer_instance):
        epoch = int(trainer_instance.epoch) + 1
        session['seconds'] = time.perf_counter() - started
        meta_path.write_text(json.dumps(meta, indent=2))
        try:
            saved = save_drive_checkpoint(config, drive_runs_dir, epoch)
        except Exception as error:
            failure = {'epoch': epoch, 'type': type(error).__name__,
                       'message': 'Drive checkpoint save failed', 'errors': _download_error_details(error)}
            (run / 'drive_backup_error.json').write_text(json.dumps(failure, indent=2))
            raise RuntimeError(f'Drive checkpoint save failed at epoch {epoch}; training stopped so unsaved epochs do not accumulate') from error
        if checkpoint_status is not None:
            checkpoint_status(saved)

    try:
        # Keep epoch progress and diagnostic output out of the notebook document.
        with local_console(run / 'console.log'):
            model = YOLO(str(checkpoint)) if resume else load_model(config)
            if drive_runs_dir is not None:
                model.add_callback('on_model_save', persist_epoch)
            if resume:
                model.train(resume=True, data=str(root / 'data.yaml'), plots=False,
                            save=True, save_period=-1, save_json=False, save_txt=False, device=device)
            else:
                model.train(data=str(root / 'data.yaml'), epochs=config.epochs, batch=config.batch,
                            imgsz=config.imgsz, seed=config.seed, deterministic=True,
                            device=device, project=str(run.parent), name=run.name, exist_ok=True,
                            patience=config.patience, workers=config.workers, cache=False,
                            single_cls=False, cos_lr=True, close_mosaic=min(10, config.epochs),
                            shear=0.0, perspective=0.0, amp=True, plots=False,
                            max_det=1000, save=True, save_period=-1,
                            save_json=False, save_txt=False, verbose=False)
        status = 'completed'
    finally:
        session.update(seconds=time.perf_counter()-started, status=status)
        meta_path.write_text(json.dumps(meta, indent=2))
    return run

if SAVE_EVERY_EPOCH_TO_DRIVE and Path(DRIVE_MOUNT_POINT).absolute() not in Path(DRIVE_CHECKPOINT_DIR).absolute().parents:
    raise ValueError('DRIVE_CHECKPOINT_DIRはマウントしたGoogle Drive配下に指定してください。')
configs = [TrainConfig(mode=mode, model=BASE_MODEL, epochs=EPOCHS, batch=BATCH_SIZE,
                       imgsz=IMAGE_SIZE, seed=SEED, debug=DEBUG_MODE, run_name=RUN_NAME,
                       allow_cpu=ALLOW_CPU) for mode in MODES_TO_RUN]
for config in configs:
    config.validate()
if not RESUME:
    with local_console(BASE / 'model_setup.log'):
        base_model = load_model(configs[0])
        del base_model
print('モデル準備完了')

## 14. 追加学習とチェックポイント管理
同じ初期モデル・seed・学習条件で各モードを学習します。結果はruns/<mode>/<RUN_NAME>です。
進捗や画像はノートブックに表示せず、ログはランタイムに保存します。T4でOOMが出たらbatchを8などに下げ、新しい実験名を使ってください。
毎epoch終了時にoptimizerを含む再開用last.ptとbest.pt・学習設定・CSVをGoogle Driveへ保存します。
保存先はDRIVE_CHECKPOINT_DIR/<mode>/<RUN_NAME>/epoch_XXXXXX_...です。SHA-256照合後に最新保存のポインタを更新します。
新しいランタイムでも、同じデータ・設定・RUN_NAMEでRESUME=Trueにして上から実行すると、Driveから復元して最後に保存できたepochの次から再開します。
再開するモードだけをMODES_TO_RUNに指定してください。目標epoch数まで完了したチェックポイントは再開対象になりません。
Drive保存に失敗した場合は学習を停止し、未保存のまま次のepochへ進みません。Drive保存中の強制終了では、その前に保存完了したepochから再開します。
本番比較はearly stoppingで実際のepoch数が異なる可能性もあるため、training CSVも確認してください。

In [ ]:
checkpoint_display = None
def show_saved_epoch(saved):
    global checkpoint_display
    from html import escape
    from IPython.display import HTML, display
    content = HTML('<pre>Drive保存完了: epoch ' + str(saved['epoch']) + '<br>' + escape(saved['directory']) + '</pre>')
    if checkpoint_display is None:
        checkpoint_display = display(content, display_id=True)
    else:
        checkpoint_display.update(content)

trained_runs = {}
for config in configs:
    trained_runs[config.mode] = train(config, resume=RESUME,
                                      drive_runs_dir=DRIVE_CHECKPOINT_DIR if SAVE_EVERY_EPOCH_TO_DRIVE else None,
                                      checkpoint_status=show_saved_epoch)
    print(config.mode, '学習完了')

## 15. 評価
各モデルをそれぞれのval正解ラベルで評価します。Precision・Recall・mAP50・mAP50-95、学習時間、推論ms/画像、重みMiBを記録します。
速度はbatch=1・FP32でのUltralytics validationのinference値です。前後処理、ブラウザ、追跡器を含むアプリ全体の速度ではありません。
学習時間は再開を含むセッション時間で、準備処理も含みます。

In [ ]:
"""Scalar-only evaluation, comparable CSVs and explicitly selected local exports."""
from dataclasses import asdict
import csv
import hashlib
import json
import shutil
from pathlib import Path


def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(1024**2), b''):
            digest.update(chunk)
    return digest.hexdigest()


def evaluate(config):
    run = config.run_dir
    best = runtime_path(run / 'weights/best.pt')
    if not best.is_file():
        raise FileNotFoundError('No trained best.pt')
    experiment = json.loads((run / 'experiment.json').read_text())
    if experiment['specification']['config'] != asdict(config):
        raise ValueError('Evaluation must use the saved training configuration')
    root = runtime_path(BASE / 'processed' / config.mode)
    validate_labels(root, config.mode)
    conversion = json.loads((root / 'conversion.json').read_text())
    if conversion['fingerprint'] != experiment['specification']['dataset_fingerprint']:
        raise ValueError('Evaluation dataset differs from the training dataset')
    YOLO = private_yolo()
    with local_console(run / 'evaluation.log'):
        model = YOLO(str(best))
        metrics = model.val(data=str(BASE / 'processed' / config.mode / 'data.yaml'),
                            split='val', imgsz=config.imgsz, batch=1, device=device_for(config),
                            workers=config.workers, seed=config.seed, conf=0.001, iou=0.7,
                            max_det=1000, half=False, plots=False, save=False,
                            save_json=False, save_txt=False, verbose=False,
                            project=str(run), name='validation', exist_ok=True)
    import torch
    identity = {'mode': config.mode, 'run_name': run.name, 'weight_sha256': sha256(best)}
    common = dict(precision=float(metrics.box.mp), recall=float(metrics.box.mr),
                  map50=float(metrics.box.map50), map50_95=float(metrics.box.map),
                  training_seconds=sum(s['seconds'] for s in experiment['sessions']),
                  inference_ms_per_image=float(metrics.speed['inference']),
                  model_mib=best.stat().st_size/1024**2,
                  device=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
    record = {**identity, **common, 'specification': experiment['specification'],
              'evaluation': {'batch': 1, 'imgsz': config.imgsz, 'half': False, 'conf': 0.001, 'iou': 0.7, 'max_det': 1000},
              'classes': []}
    for index, class_id in enumerate(metrics.box.ap_class_index):
        p, r, ap50, ap = metrics.box.class_result(index)
        record['classes'].append({'class_id': int(class_id), 'name': model.names[int(class_id)],
                                  'precision': float(p), 'recall': float(r),
                                  'map50': float(ap50), 'map50_95': float(ap)})
    (run / 'evaluation.json').write_text(json.dumps(record, indent=2))
    return {**identity, **common}


def comparison(output=BASE / 'exports/comparison.csv'):
    rows, signatures = [], []
    for path in sorted((BASE / 'runs').glob('*/*/evaluation.json')):
        runtime_path(path)
        row = json.loads(path.read_text())
        best = runtime_path(path.parent / 'weights/best.pt')
        if not best.is_file() or sha256(best) != row['weight_sha256']:
            raise ValueError('Stale evaluation: evaluate the current checkpoint again')
        spec = json.loads(json.dumps(row['specification']))
        for key in ('mode', 'run_name'):
            spec['config'].pop(key)
        signatures.append(json.dumps({'spec': spec, 'device': row['device'], 'evaluation': row['evaluation']}, sort_keys=True))
        scalar = {k: v for k, v in row.items() if k not in ('specification', 'evaluation', 'classes')}
        rows.append({**scalar, 'scope': 'all', 'class_name': 'all'})
        for cls in row['classes']:
            rows.append({**scalar, **{k: cls[k] for k in ('precision', 'recall', 'map50', 'map50_95')},
                         'scope': 'class', 'class_name': cls['name']})
    if not rows:
        raise FileNotFoundError('No evaluation results; train and evaluate at least one mode')
    comparable = len(set(signatures)) == 1
    for row in rows:
        row['matched_conditions'] = comparable
    output = runtime_path(output)
    with open(output, 'w', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    return rows


def inspect_onnx(path, image_size, contract=None):
    """Run synthetic input only. A supplied application contract is required for matching."""
    import numpy as np
    import onnx
    import onnxruntime as ort
    path = runtime_path(path)
    onnx.checker.check_model(onnx.load(str(path)))
    session = ort.InferenceSession(str(path), providers=['CPUExecutionProvider'])
    inputs = session.get_inputs()
    if len(inputs) != 1 or inputs[0].type != 'tensor(float)' or len(inputs[0].shape) != 4:
        raise ValueError('Unexpected ONNX image input')
    tensor = np.zeros((1, 3, image_size, image_size), dtype=np.float32)
    outputs = session.run(None, {inputs[0].name: tensor})
    if not outputs or any(not np.isfinite(value).all() for value in outputs):
        raise ValueError('Invalid ONNX synthetic inference output')
    interface = {'input': {'name': inputs[0].name, 'dtype': inputs[0].type, 'shape': list(tensor.shape)},
                 'outputs': [{'name': meta.name, 'dtype': meta.type, 'shape': list(value.shape)}
                             for meta, value in zip(session.get_outputs(), outputs)]}
    if contract is not None and interface != contract:
        raise ValueError('Web application input/output contract mismatch')
    return {'interface': interface, 'onnx_checker': 'passed', 'synthetic_inference': 'passed',
            'application_contract': 'matched' if contract is not None else 'unverified',
            'browser_integration': 'unverified',
            'preprocessing': 'RGB float32 NCHW /255; letterbox and inverse coordinate mapping required',
            'postprocessing': 'Confirm YOLO26 end-to-end output vs raw detections before connecting tracker'}


def export_model(config, onnx_export=True, contract=None):
    YOLO = private_yolo()
    best = runtime_path(config.run_dir / 'weights/best.pt')
    if not best.is_file():
        raise FileNotFoundError('No trained best.pt')
    prefix = f'{config.mode}_{config.run_dir.name}'
    out = BASE / 'exports'
    weight = runtime_path(out / f'{prefix}.pt')
    shutil.copyfile(best, weight)
    csv_source = runtime_path(config.run_dir / 'results.csv')
    if csv_source.is_file():
        shutil.copyfile(csv_source, out / f'{prefix}_training.csv')
    if onnx_export:
        with local_console(config.run_dir / 'export.log'):
            model = YOLO(str(weight))
            exported = runtime_path(model.export(format='onnx', imgsz=config.imgsz, opset=13,
                                                 dynamic=True, simplify=False, nms=False,
                                                 half=False, batch=1, device='cpu'))
            report = inspect_onnx(exported, config.imgsz, contract)
        report['names'] = model.names
        report['weight_sha256'] = sha256(weight)
        (out / f'{prefix}_onnx_interface.json').write_text(json.dumps(report, indent=2))
    # Deliberately no zip of the run directory or image/annotation inclusion.
    return [p.name for p in sorted(out.glob(prefix + '*')) if p.suffix in ('.pt', '.onnx', '.csv')]


def download_selected(names):
    """Called only from an opt-in notebook cell; no external publication."""
    from google.colab import files
    for name in names:
        if Path(name).name != name or Path(name).suffix not in ('.pt', '.onnx', '.csv'):
            raise ValueError('Only explicitly selected pt/onnx/csv exports may be downloaded')
        path = runtime_path(BASE / 'exports' / name)
        if not path.is_file():
            raise FileNotFoundError('Selected export does not exist')
        files.download(str(path))

for config in configs:
    print(evaluate(config))

## 16. vbox・hboxの比較
検出対象が異なるためmAPだけで優劣は判定できません。bothではクラス別の行も出力します。
複数モードを実行するか、設定のモードを切り替えて学習・評価を追加後、このセルを再実行してください。
matched_conditionsは保存した設定・データ指紋・依存・評価条件・GPU名の一致を表します。対象の同一性や完全な再現性を保証する値ではありません。
重みが変わった後の古い評価結果はエラーになります。

In [ ]:
comparison_rows = comparison()
print({'comparison_csv': f'{EXPORT_DIR}/comparison.csv', 'rows': len(comparison_rows),
       'matched_conditions': all(row['matched_conditions'] for row in comparison_rows)})
for row in comparison_rows:
    if row['scope'] == 'all':
        print(row)

## 17. モデルのエクスポート
exportsに.pt・ONNX・学習CSVを出力します。画像・ラベル・run一式のZIPは作成しません。
ONNXはopset 13・dynamic・FP32、simplify=Falseで出力し、checkerと合成入力によるONNX Runtime CPU推論を実行します。
入出力名・shape・dtype・クラス定義は*_onnx_interface.jsonに記録します。YOLO26のend-to-end出力は、従来のYOLOの生出力を想定するアプリと異なる場合があります。
WEBAPP_CONTRACTに実アプリの入出力辞書を設定すると完全一致を検査します。未指定は互換性未検証です。
RGB/NCHW/0〜1、letterbox・座標逆変換、confidence・クラスID・NMS有無、ONNX Runtime Web対応opsetをアプリ側で検証してください。

In [ ]:
exported_names = []
for config in configs:
    exported_names.extend(export_model(config, onnx_export=EXPORT_ONNX, contract=WEBAPP_CONTRACT))
print({'exported': exported_names})

## 18. 必要ファイルの明示ダウンロード
このセルは初期状態ではダウンロードしません。必要なファイル名をDOWNLOAD_NAMESに列挙し、DO_DOWNLOAD=Trueに変更して、このセルだけを明示的に実行してください。
.pt・.onnx・.csvに限定します。公開操作ではありません。学習済みモデルにもデータ由来の情報が含まれる可能性があるため、外部公開・商用利用前に条件を確認してください。

In [ ]:
DO_DOWNLOAD = False
DOWNLOAD_NAMES = []  # 例: ['vbox_experiment.pt', 'comparison.csv']
if DO_DOWNLOAD:
    download_selected(DOWNLOAD_NAMES)
else:
    print('ダウンロード未実行')

## 19. 一時データの削除（任意）
学習済みモデルの取得を確認してから実行してください。初期状態では削除しません。
DELETE_DATA=Trueはrawとprocessedを削除します。DELETE_RUNS=TrueはColab内のチェックポイントと学習ログを削除します。Drive側の保存は残るため、RESUME=Trueで復元できます。
DEBUGから本番へ切り替える際は、DELETE_DATAを使うとrawも消えます。rawを再利用する場合はColabファイル欄でprocessed/<mode>だけを明示的に削除し、新しいRUN_NAMEで変換・学習してください。
exportsは残ります。ランタイムのリセットではexportsも含めて消失します。
このセルはDriveの保管フォルダを削除しません。次のランタイムではDriveから再コピーできます。

In [ ]:
DELETE_DATA = False
DELETE_RUNS = False
import shutil
if DELETE_DATA:
    for part in ('raw', 'processed'):
        target = runtime_path(BASE / part)
        if target.exists():
            shutil.rmtree(target)
if DELETE_RUNS:
    target = runtime_path(BASE / 'runs')
    if target.exists():
        shutil.rmtree(target)
print('削除設定に従って処理しました')